# 3D Face Reconstruction with Occlusion

Converted from [`new_(3) (8).py`](../new_%283%29%20%288%29.py) in the `3DFaceReconstruction-With-Occlusion` repository.

The exported script did not contain reliable notebook cell-boundary markers, so its code is preserved in one ordered code cell. The standalone `pip install` command has been changed to `!pip install` for IPython/Colab compatibility. Runtime execution has not been performed as part of this conversion.


In [ ]:
# -*- coding: utf-8 -*-
"""new (3).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1abCNo7JKM206Uq66kmWIShO3rDqkE78S
"""

# Commented out IPython magic to ensure Python compatibility.
# -------------------------------
# 1. Mount Google Drive FIRST
# -------------------------------
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# 2. Clone main repo (force re-clone for robustness)
# -------------------------------
import os

# Remove existing directory to ensure a clean clone
if os.path.exists('/content/Deep3DFaceRecon_pytorch'):
    !rm -rf /content/Deep3DFaceRecon_pytorch

!git clone https://github.com/sicxu/Deep3DFaceRecon_pytorch.git

# %cd /content/Deep3DFaceRecon_pytorch

# -------------------------------
# 3. Install dependencies
# -------------------------------
!pip install torch torchvision
!pip install kornia dominate trimesh
!pip install opencv-python scikit-image tqdm
!pip install yacs chumpy face-alignment

# -------------------------------
# 4. Install nvdiffrast (from Drive - persistent)
# -------------------------------
# %cd /content/drive/MyDrive/nvdiffrast

# Ensure correct folder (important)
!ls

!pip install setuptools wheel ninja
!pip install . --no-build-isolation

# -------------------------------
# 5. Go back to project
# -------------------------------
# %cd /content/Deep3DFaceRecon_pytorch

# -------------------------------
# 6. Clone insightface (only once)
# -------------------------------
!git clone https://github.com/deepinsight/insightface.git
!cp -r ./insightface/recognition/arcface_torch ./models/

# -------------------------------
# 7. Setup BFM folder
# -------------------------------
!mkdir -p BFM

!cp /content/drive/MyDrive/BFM/01_MorphableModel.mat BFM/
!cp /content/drive/MyDrive/BFM/Exp_Pca.bin BFM/

# -------------------------------
# DONE ✅
# -------------------------------
print("✅ Setup complete!")

# Commented out IPython magic to ensure Python compatibility.
# %cd /content/Deep3DFaceRecon_pytorch

!mkdir -p checkpoints

!cp -r /content/drive/MyDrive/face_recon_feat0.2_augment checkpoints/

!ls checkpoints/face_recon_feat0.2_augment

!ls /content/Deep3DFaceRecon_pytorch/datasets/examples

# Commented out IPython magic to ensure Python compatibility.
# %cd /content/Deep3DFaceRecon_pytorch
!mkdir -p my_images/detections

# Commented out IPython magic to ensure Python compatibility.
# %cd /content/Deep3DFaceRecon_pytorch/util

!sed -i 's/warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)/warnings.filterwarnings("ignore")/g' preprocess.py
!sed -i 's/trans_params = np.array(\[w0, h0, s, t\[0\], t\[1\]\])/trans_params = np.array([float(w0), float(h0), float(s), float(t[0]), float(t[1])])/g' preprocess.py

# Commented out IPython magic to ensure Python compatibility.
# %cd /content/Deep3DFaceRecon_pytorch

!python test.py \
  --name=face_recon_feat0.2_augment \
  --epoch=20 \
  --img_folder=./datasets/examples

!ls checkpoints/face_recon_feat0.2_augment/results/*/*

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Path to results folder
RESULTS_DIR = 'checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000'

# Loop through all result images
for file in os.listdir(RESULTS_DIR):

    if file.endswith('.png'):

        print(f"\nProcessing: {file}")

        img_path = os.path.join(RESULTS_DIR, file)
        img_bgr = cv2.imread(img_path)

        if img_bgr is None:
            print("❌ Failed to load image")
            continue

        # Split columns
        h, w, _ = img_bgr.shape
        img_orig = img_bgr[:, 0:h]
        img_render = img_bgr[:, h:h*2]

        # Mask (remove black background)
        gray = cv2.cvtColor(img_render, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 5, 255, cv2.THRESH_BINARY)

        orig_masked = cv2.bitwise_and(img_orig, img_orig, mask=mask)
        render_masked = cv2.bitwise_and(img_render, img_render, mask=mask)

        # Convert to grayscale
        orig_gray = orig_masked.astype(np.float32)
        render_gray = render_masked.astype(np.float32)
        diff = np.mean(np.abs(orig_gray - render_gray), axis=2)

        # Normalize
        diff_norm = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        # Heatmap
        heatmap = cv2.applyColorMap(diff_norm, cv2.COLORMAP_JET)
        heatmap = cv2.bitwise_and(heatmap, heatmap, mask=mask)

        # Show
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(cv2.cvtColor(orig_masked, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original")
        axes[0].axis('off')

        axes[1].imshow(cv2.cvtColor(render_masked, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Rendered")
        axes[1].axis('off')

        axes[2].imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
        axes[2].set_title("Error Heatmap")
        axes[2].axis('off')

        plt.show()

!pip install pyglet



from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000/vd051.png')

plt.imshow(img)
plt.axis('off')

import trimesh

mesh = trimesh.load('checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000/vd051.obj')

mesh.show()

!pip install opencv-python scikit-image numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from PIL import Image

# 1. Load the combined output image that Deep3DFaceRecon created
result_image_path = 'checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000/vd051.png'
img_bgr = cv2.imread(result_image_path)

if img_bgr is None:
    print(f"Error: Could not load {result_image_path}")
else:
    # 2. Split the image columns
    # Deep3DFaceRecon natively outputs 224x224 squares side-by-side.
    h, w, c = img_bgr.shape
    num_images = w // h # Usually w=672 and h=224, meaning 3 images combined

    # Crop Column 0 (Original Input) and Column 1 (Reconstructed Render)
    img_orig = img_bgr[:, 0:h]
    img_render = img_bgr[:, h:h*2] # The textured 3D mesh rendered to 2D

    # 3. Create a mask so we only compare the face (ignoring the black background)
    gray_render = cv2.cvtColor(img_render, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray_render, 5, 255, cv2.THRESH_BINARY)

    img_orig_masked = cv2.bitwise_and(img_orig, img_orig, mask=mask)
    img_render_masked = cv2.bitwise_and(img_render, img_render, mask=mask)

    # 4. Compute Accuracy Metrics
    error_mse = mse(img_orig_masked, img_render_masked)
    error_psnr = psnr(img_orig_masked, img_render_masked, data_range=255)
    error_ssim, _ = ssim(img_orig_masked, img_render_masked,
                         channel_axis=-1,
                         full=True,
                         data_range=255,
                         win_size=7)

    # 5. Print out the Accuracy
    print("=== 3D Reconstruction Accuracy (2D Render Metric) ===")
    print(f"MSE  (Mean Squared Error): {error_mse:.2f}")
    print(f"PSNR (Peak Signal-to-Noise): {error_psnr:.2f} dB")
    print(f"SSIM (Structural Similarity): {error_ssim:.4f}")

    # 6. Plotting to double-check
    img_orig_masked_rgb = cv2.cvtColor(img_orig_masked, cv2.COLOR_BGR2RGB)
    img_render_masked_rgb = cv2.cvtColor(img_render_masked, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img_orig_masked_rgb)
    axes[0].set_title("Input Image (Masked)")
    axes[0].axis('off')

    axes[1].imshow(img_render_masked_rgb)
    axes[1].set_title("Rendered 3D Mesh (Masked)")
    axes[1].axis('off')
    plt.show()

import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Ensure you still have your masked images from the previous cell
# img_orig_masked and img_render_masked

# 2. Convert to Grayscale or float for accurate difference
orig_gray = cv2.cvtColor(img_orig_masked, cv2.COLOR_BGR2GRAY).astype(np.float32)
render_gray = cv2.cvtColor(img_render_masked, cv2.COLOR_BGR2GRAY).astype(np.float32)

# 3. Calculate the Absolute Difference (Error Map)
# This subtracts the pixels from one another. 0 = perfect match, 255 = total failure.
diff_map = np.abs(orig_gray - render_gray)

# 4. Normalize the error map to [0, 255] for visualization
diff_map_norm = cv2.normalize(diff_map, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
diff_map_norm = diff_map_norm.astype(np.uint8)

# 5. Apply a Color Map (Heatmap)
# Blue/Dark = Low Error, Red/Yellow = High Error
heatmap_img = cv2.applyColorMap(diff_map_norm, cv2.COLORMAP_JET)

# We only want to see the heatmap on the face, so we apply the mask again
heatmap_masked = cv2.bitwise_and(heatmap_img, heatmap_img, mask=mask)

# 6. Plot the results
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

axes[0].imshow(cv2.cvtColor(img_orig_masked, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Face")
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_render_masked, cv2.COLOR_BGR2RGB))
axes[1].set_title("Reconstructed Render")
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(heatmap_masked, cv2.COLOR_BGR2RGB))
axes[2].set_title("Error Heatmap (Red = High Error)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

# 7. Print analysis
print(f"Max Pixel Difference: {np.max(diff_map):.2f}")
print(f"Average Pixel Difference in face: {np.mean(diff_map[mask > 0]):.2f}")

# Commented out IPython magic to ensure Python compatibility.
# Clone the BiSeNet face parsing repository
!git clone https://github.com/zllrunning/face-parsing.PyTorch
# %cd face-parsing.PyTorch
!mkdir -p res/cp

# Download the pre-trained weights
!gdown --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O res/cp/79999_iter.pth

# %cd /content/Deep3DFaceRecon_pytorch

import os, shutil, sys
import torch, cv2, numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from PIL import Image
from skimage.metrics import structural_similarity as ssim

# ── 1. Device and Paths ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sys.path.insert(0, '/content/Deep3DFaceRecon_pytorch/face-parsing.PyTorch')
from model import BiSeNet

# ── 2. Download Weights ──
weights_path = '/content/Deep3DFaceRecon_pytorch/face-parsing.PyTorch/res/cp/79999_iter.pth'
os.makedirs(os.path.dirname(weights_path), exist_ok=True)

if not os.path.exists(weights_path) or os.path.getsize(weights_path) < 50_000_000:
    from huggingface_hub import hf_hub_download
    cached = hf_hub_download(repo_id="vivym/face-parsing-bisenet", filename="79999_iter.pth")
    shutil.copy(cached, weights_path)
    print(f'✓ Weights Downloaded: {os.path.getsize(weights_path)/1e6:.1f} MB')

# ── 3. Load BiSeNet ──
net = BiSeNet(n_classes=19)
state = torch.load(weights_path, map_location=device)

# Fix DataParallel keys if present
if all(k.startswith('module.') for k in state.keys()):
    state = {k[7:]: v for k, v in state.items()}

net.load_state_dict(state)
net.to(device)
net.eval()

print('✓ BiSeNet loaded successfully!')

# ── 4. Transform ──
bisenet_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Face labels (exclude hair, background)
FACE_LABELS = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

# ── 5. Segmentation ──
@torch.no_grad()
def get_segmentation(net, img_bgr, device):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inp = bisenet_transform(pil).unsqueeze(0).to(device)

    out = net(inp)[0]
    seg = out.squeeze(0).argmax(0).cpu().numpy().astype(np.uint8)

    seg = cv2.resize(seg, (img_bgr.shape[1], img_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    return seg

def make_face_mask(seg_map):
    mask = np.zeros_like(seg_map, dtype=np.uint8)

    for lbl in FACE_LABELS:
        mask[seg_map == lbl] = 1

    # Morphological smoothing
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

# ── 6. Generate Mask ──
seg_map = get_segmentation(net, img_orig, device)
mask = make_face_mask(seg_map).astype(bool)

# ── 7. CORRECT MASKED MSE ──
img_orig_f = img_orig.astype(np.float32)
img_render_f = img_render.astype(np.float32)

diff = (img_orig_f - img_render_f) ** 2
masked_diff = diff[mask]

error_mse = masked_diff.mean()

# ── 8. CORRECT SSIM (crop to valid region) ──
ys, xs = np.where(mask)

ymin, ymax = ys.min(), ys.max()
xmin, xmax = xs.min(), xs.max()

crop_orig = img_orig[ymin:ymax, xmin:xmax]
crop_render = img_render[ymin:ymax, xmin:xmax]

error_ssim = ssim(
    crop_orig,
    crop_render,
    channel_axis=-1,
    data_range=255
)

# ── 9. Visualization ──
mask_vis = (mask * 255).astype(np.uint8)

img_orig_masked = cv2.bitwise_and(img_orig, img_orig, mask=mask_vis)
img_render_masked = cv2.bitwise_and(img_render, img_render, mask=mask_vis)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(mask_vis, cmap='gray')
axes[1].set_title("Face Mask")
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(img_orig_masked, cv2.COLOR_BGR2RGB))
axes[2].set_title("Original (Face Only)")
axes[2].axis('off')

axes[3].imshow(cv2.cvtColor(img_render_masked, cv2.COLOR_BGR2RGB))
axes[3].set_title("Rendered (Face Only)")
axes[3].axis('off')

plt.show()

# ── 10. Debug Info ──
print("\n--- DEBUG INFO ---")
print("Total pixels:", img_orig.shape[0] * img_orig.shape[1])
print("Face pixels used:", mask.sum())

# ── 11. Final Metrics ──
print("\n--- FINAL CORRECTED METRICS ---")
print(f"Masked MSE:  {error_mse:.2f}")
print(f"Masked SSIM: {error_ssim:.4f}")

!pip install insightface onnxruntime-gpu -q

!pip install mtcnn

import os
import cv2
from mtcnn import MTCNN

# 1. Define your image folder
image_folder = '/content/drive/MyDrive/my_images'  # Change this if your images are located elsewhere
detections_folder = os.path.join(image_folder, 'detections')
os.makedirs(detections_folder, exist_ok=True)

# 2. Initialize MTCNN detector
detector = MTCNN()

# 3. Process each image in the folder
for filename in os.listdir(image_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(image_folder, filename)

        # Read image
        img = cv2.imread(img_path)
        if img is None:
            continue

        # MTCNN requires RGB images
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Detect face(s)
        results = detector.detect_faces(img_rgb)

        if len(results) > 0:
            # We take the first (most prominent) face detected
            keypoints = results[0]['keypoints']

            # Deep3DFaceRecon expects 5 landmarks in this exact order:
            # Left Eye, Right Eye, Nose, Mouth Left, Mouth Right
            landmarks = [
                keypoints['left_eye'],
                keypoints['right_eye'],
                keypoints['nose'],
                keypoints['mouth_left'],
                keypoints['mouth_right']
            ]

            # Create a .txt file with the exact same name as the image
            txt_filename = os.path.splitext(filename)[0] + '.txt'
            txt_path = os.path.join(detections_folder, txt_filename)

            # Write coordinates to the text file
            with open(txt_path, 'w') as f:
                for pt in landmarks:
                    f.write(f"{pt[0]} {pt[1]}\n")

            print(f"✅ Saved landmarks for {filename}")
        else:
            print(f"❌ No face detected in {filename}")

img_folder = '/content/drive/MyDrive/my_images'

import os
import numpy as np

img_folder = '/content/drive/MyDrive/my_images'  # FIXED PATH
img_name = 'negi.png'

det_path = os.path.join(img_folder, 'detections', 'negi.txt')

print("=== 1. Checking File Existence ===")
img_exists = os.path.exists(os.path.join(img_folder, img_name))
det_exists = os.path.exists(det_path)
print(f"Image {img_name} found: {img_exists}")
print(f"Text file henry.txt found: {det_exists}")

print("\n=== 2. Checking Text Content ===")
if det_exists:
    try:
        landmarks = np.loadtxt(det_path)
        print(f"Successfully loaded landmarks with shape: {landmarks.shape}")
        print(landmarks)
    except Exception as e:
        print(f"FAILED to read landmarks. Error: {e}")

print("\n=== 3. Checking Deep3D Internal Path Loader ===")
expected_lm_path = os.path.join(
    img_folder,
    'detections',
    img_name.replace('.png', '.txt').replace('.jpg', '.txt')
)
print(f"Deep3D is searching for text file at: {expected_lm_path}")
print(f"Does Deep3D see this as a valid file? -> {os.path.isfile(expected_lm_path)}")

# ===== OCCLUSION MODULE START =====
# Inserts occlusion handling BEFORE Deep3DFaceRecon test-time reconstruction.
import os, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch # Added for device
from PIL import Image # Added for get_segmentation
import torchvision.transforms as transforms # Added for get_segmentation

RECON_ROOT = '/content/Deep3DFaceRecon_pytorch'
INPUT_IMG_FOLDER = '/content/drive/MyDrive/my_images'
INPAINTED_FOLDER_REL = 'input_images_inpainted'
INPAINTED_IMG_FOLDER = os.path.join(RECON_ROOT, INPAINTED_FOLDER_REL)

# Keep relative paths consistent with the later `!python test.py ...` cell.
os.makedirs(INPAINTED_IMG_FOLDER, exist_ok=True)
os.makedirs(os.path.join(INPAINTED_IMG_FOLDER, 'detections'), exist_ok=True)
os.chdir(RECON_ROOT)

# --- Explicitly redefine BiSeNet helpers here for robustness against stale kernel state ---
# These are copied from cell 01ZM-x1x3mHb to ensure correct definitions are used.

# Ensure BiSeNet model 'net' and device are loaded from cell 01ZM-x1x3mHb
# (Assuming `net`, `device`, `bisenet_transform`, `FACE_LABELS` are in scope from 01ZM-x1x3mHb)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Redefine device for robustness

bisenet_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
FACE_LABELS = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

@torch.no_grad()
def get_segmentation(net, img_bgr, device):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inp = bisenet_transform(pil).unsqueeze(0).to(device)

    out = net(inp)[0]
    seg = out.squeeze(0).argmax(0).cpu().numpy().astype(np.uint8)

    seg = cv2.resize(seg, (img_bgr.shape[1], img_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    return seg

def make_face_mask(seg_map):
    mask = np.zeros_like(seg_map, dtype=np.uint8)

    for lbl in FACE_LABELS:
        mask[seg_map == lbl] = 1

    # Morphological smoothing
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

# Face occlusion mask = NOT(face_mask)
def inpaint_face(image_bgr, occlusion_mask_uint8_0_255):
    # OpenCV expects: single-channel uint8 mask where non-zero indicates regions to inpaint.
    return cv2.inpaint(image_bgr, occlusion_mask_uint8_0_255, 3, cv2.INPAINT_TELEA)

# --- Preprocess ALL input images (segmentation -> occlusion mask -> inpainting) ---
image_exts = ('.png', '.jpg', '.jpeg', '.bmp')
image_files = [f for f in sorted(os.listdir(INPUT_IMG_FOLDER)) if f.lower().endswith(image_exts)]

# Copy landmark txts so Deep3DFaceRecon keeps the same correspondence.
src_det_dir = os.path.join(INPUT_IMG_FOLDER, 'detections')
dst_det_dir = os.path.join(INPAINTED_IMG_FOLDER, 'detections')
if os.path.exists(src_det_dir):
    for fn in os.listdir(src_det_dir):
        if fn.lower().endswith('.txt'):
            shutil.copy(os.path.join(src_det_dir, fn), os.path.join(dst_det_dir, fn))

example_to_visualize = 'manbhav_image.jpg'

for idx, filename in enumerate(image_files):
    img_path = os.path.join(INPUT_IMG_FOLDER, filename)
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f'⚠️ Skipping unreadable image: {filename}')
        continue

    # Step 1: Face segmentation -> face mask
    seg_map = get_segmentation(net, img_bgr, device)
    face_mask = make_face_mask(seg_map).astype(bool)

    # Step 2: Occlusion mask (IMPORTANT FIX)
    # occlusion_mask = NOT(face_mask)
    occlusion_mask = (~face_mask).astype(np.uint8) * 255  # 0 or 255

    # slight smoothing (morphology) to reduce speckles
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    occlusion_mask = cv2.morphologyEx(occlusion_mask, cv2.MORPH_OPEN, kernel)
    occlusion_mask = cv2.morphologyEx(occlusion_mask, cv2.MORPH_CLOSE, kernel)

    # Step 3: Inpainting (MANDATORY)
    img_inpainted_bgr = inpaint_face(img_bgr, occlusion_mask)

    # Save inpainted image with the SAME filename
    out_path = os.path.join(INPAINTED_IMG_FOLDER, filename)
    cv2.imwrite(out_path, img_inpainted_bgr)

    # Step 5: Visualization (for debugging)
    if filename == example_to_visualize:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        inpaint_rgb = cv2.cvtColor(img_inpainted_bgr, cv2.COLOR_BGR2RGB)

        face_mask_vis = (face_mask.astype(np.uint8) * 255)
        occlusion_mask_vis = occlusion_mask

        plt.figure(figsize=(16, 4))
        plt.subplot(1, 4, 1)
        plt.title('Original')
        plt.imshow(img_rgb)
        plt.axis('off')

        plt.subplot(1, 4, 2)
        plt.title('Face mask')
        plt.imshow(face_mask_vis, cmap='gray')
        plt.axis('off')

        plt.subplot(1, 4, 3)
        plt.title('Occlusion mask')
        plt.imshow(occlusion_mask_vis, cmap='gray')
        plt.axis('off')

        plt.subplot(1, 4, 4)
        plt.title('Inpainted')
        plt.imshow(inpaint_rgb)
        plt.axis('off')
        plt.show()

print('✅ Inpainting preprocessing complete. Feeding inpainted images to Deep3DFaceRecon...')
# ===== OCCLUSION MODULE END =====

!python test.py \
  --name=face_recon_feat0.2_augment \
  --epoch=20 \
  --img_folder=input_images_inpainted \
  --dataset_mode= \
  --phase=test

"""### 3D Reconstruction for Original (Non-Inpainted) Images

To compare, we need to run the reconstruction process again, but this time using the original input images without any inpainting. We'll set up a temporary folder for these images and their detection files.
"""

# Commented out IPython magic to ensure Python compatibility.
import os, shutil

# Define paths
RECON_ROOT = '/content/Deep3DFaceRecon_pytorch'
INPUT_IMG_FOLDER = '/content/drive/MyDrive/my_images'
TEMP_ORIG_INPUT_FOLDER_NAME = 'original_images_for_recon'
TEMP_ORIG_INPUT_FOLDER = os.path.join(RECON_ROOT, TEMP_ORIG_INPUT_FOLDER_NAME)
TEMP_ORIG_DETECTIONS_FOLDER = os.path.join(TEMP_ORIG_INPUT_FOLDER, 'detections')

# Create temporary directories
os.makedirs(TEMP_ORIG_INPUT_FOLDER, exist_ok=True)
os.makedirs(TEMP_ORIG_DETECTIONS_FOLDER, exist_ok=True)

# Copy original images to the temporary folder
image_exts = ('.png', '.jpg', '.jpeg', '.bmp')
for filename in os.listdir(INPUT_IMG_FOLDER):
    if filename.lower().endswith(image_exts):
        shutil.copy(os.path.join(INPUT_IMG_FOLDER, filename), os.path.join(TEMP_ORIG_INPUT_FOLDER, filename))

# Copy landmark detection files to the temporary detections folder
s_det_dir = os.path.join(INPUT_IMG_FOLDER, 'detections')
if os.path.exists(s_det_dir):
    for fn in os.listdir(s_det_dir):
        if fn.lower().endswith('.txt'):
            shutil.copy(os.path.join(s_det_dir, fn), os.path.join(TEMP_ORIG_DETECTIONS_FOLDER, fn))

print(f'✅ Copied original images to {TEMP_ORIG_INPUT_FOLDER}')
print(f'✅ Copied landmark detections to {TEMP_ORIG_DETECTIONS_FOLDER}')

# Change directory to Deep3DFaceRecon_pytorch for test.py command
# %cd /content/Deep3DFaceRecon_pytorch

!python test.py \
  --name=face_recon_feat0.2_augment \
  --epoch=20 \
  --img_folder=original_images_for_recon \
  --dataset_mode= \
  --phase=test

"""### Visualize 2D Render and 3D Object for Original `manbhav_image.jpg`

Let's visualize the 2D render and the 3D mesh generated from the *original* `manbhav_image.jpg` (without inpainting).
"""

from PIL import Image
import matplotlib.pyplot as plt
import trimesh

# Define paths for the original image results
ORIGINAL_IMAGE_SOURCE = '/content/drive/MyDrive/my_images/manbhav_image.jpg' # Original source file

# Construct the output paths for Deep3DFaceRecon's results for the original image
result_folder_original = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/original_images_for_recon/epoch_20_000000'
output_image_name_original = os.path.splitext(os.path.basename(ORIGINAL_IMAGE_SOURCE))[0] + '.png'
output_obj_name_original = os.path.splitext(os.path.basename(ORIGINAL_IMAGE_SOURCE))[0] + '.obj'

img_path_original = os.path.join(result_folder_original, output_image_name_original)
mesh_path_original = os.path.join(result_folder_original, output_obj_name_original)


# Display 2D render
if os.path.exists(img_path_original):
    print(f"Displaying 2D render for: {output_image_name_original}")
    img_original = Image.open(img_path_original)
    plt.imshow(img_original)
    plt.axis('off')
    plt.title(f'2D Render (Original: {output_image_name_original})')
    plt.show()
else:
    print(f"Error: 2D render not found at {img_path_original}")

# Display 3D mesh (this will open an interactive viewer if pyglet is installed)
if os.path.exists(mesh_path_original):
    print(f"Displaying 3D mesh for: {output_obj_name_original}")
    mesh_original = trimesh.load(mesh_path_original)
    mesh_original.show()
else:
    print(f"Error: 3D mesh not found at {mesh_path_original}")

"""### Metric Analysis for Original `manbhav_image.jpg` (Without Inpainting)

Now, let's calculate the accuracy metrics (MSE, PSNR, SSIM) for the 3D reconstruction of the *original* `manbhav_image.jpg`. We will use BiSeNet for more accurate face masking for the comparison.
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from PIL import Image
import os

# Define the path to the original image result from Deep3DFaceRecon
result_image_path_original = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/original_images_for_recon/epoch_20_000000/manbhav_image.png'
img_bgr_original = cv2.imread(result_image_path_original)

if img_bgr_original is None:
    print(f"Error: Could not load {result_image_path_original}")
else:
    # Split the image columns (Original Input and Reconstructed Render)
    h, w, c = img_bgr_original.shape
    img_orig_analysis_original = img_bgr_original[:, 0:h]      # The original input image
    img_render_analysis_original = img_bgr_original[:, h:h*2] # The textured 3D mesh rendered to 2D

    # Generate Mask using BiSeNet for this specific image
    # The 'net', 'get_segmentation', 'make_face_mask', 'device' are assumed to be in scope.
    seg_map_analysis_original = get_segmentation(net, img_orig_analysis_original, device)
    face_mask_bool_analysis_original = make_face_mask(seg_map_analysis_original).astype(bool)

    # CORRECT MASKED MSE
    img_orig_f_analysis_original = img_orig_analysis_original.astype(np.float32)
    img_render_f_analysis_original = img_render_analysis_original.astype(np.float32)

    diff_analysis_original = (img_orig_f_analysis_original - img_render_f_analysis_original) ** 2
    masked_diff_analysis_original = diff_analysis_original[face_mask_bool_analysis_original]

    error_mse_analysis_original = masked_diff_analysis_original.mean()

    # CORRECT PSNR (Need to handle masked regions carefully)
    # Create temporary images with masked-out regions set to 0 for PSNR calculation
    mask_for_psnr = face_mask_bool_analysis_original.astype(np.uint8) * 255
    temp_orig_for_psnr = cv2.bitwise_and(img_orig_analysis_original, img_orig_analysis_original, mask=mask_for_psnr)
    temp_render_for_psnr = cv2.bitwise_and(img_render_analysis_original, img_render_analysis_original, mask=mask_for_psnr)

    # Ensure data_range is correct for PSNR calculation
    error_psnr_analysis_original = psnr(temp_orig_for_psnr, temp_render_for_psnr, data_range=255)

    # CORRECT SSIM (crop to valid region)
    ys_analysis_original, xs_analysis_original = np.where(face_mask_bool_analysis_original)

    ymin_analysis_original, ymax_analysis_original = ys_analysis_original.min(), ys_analysis_original.max()
    xmin_analysis_original, xmax_analysis_original = xs_analysis_original.min(), xs_analysis_original.max()

    crop_orig_analysis_original = img_orig_analysis_original[ymin_analysis_original:ymax_analysis_original, xmin_analysis_original:xmax_analysis_original]
    crop_render_analysis_original = img_render_analysis_original[ymin_analysis_original:ymax_analysis_original, xmin_analysis_original:xmax_analysis_original]

    error_ssim_analysis_original = ssim(
        crop_orig_analysis_original,
        crop_render_analysis_original,
        channel_axis=-1,
        data_range=255
    )

    # Visualization
    mask_vis_original = (face_mask_bool_analysis_original * 255).astype(np.uint8)

    img_orig_masked_original = cv2.bitwise_and(img_orig_analysis_original, img_orig_analysis_original, mask=mask_vis_original)
    img_render_masked_original = cv2.bitwise_and(img_render_analysis_original, img_render_analysis_original, mask=mask_vis_original)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(cv2.cvtColor(img_orig_analysis_original, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original Input ({os.path.basename(result_image_path_original)})")
    axes[0].axis('off')

    axes[1].imshow(mask_vis_original, cmap='gray')
    axes[1].set_title("Face Mask (BiSeNet)")
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(img_orig_masked_original, cv2.COLOR_BGR2RGB))
    axes[2].set_title("Original (Face Only)")
    axes[2].axis('off')

    axes[3].imshow(cv2.cvtColor(img_render_masked_original, cv2.COLOR_BGR2RGB))
    axes[3].set_title("Rendered (Face Only)")
    axes[3].axis('off')

    plt.show()

    # Debug Info & Final Metrics
    print("\n--- DEBUG INFO FOR ORIGINAL IMAGE ---")
    print("Total pixels:", img_orig_analysis_original.shape[0] * img_orig_analysis_original.shape[1])
    print("Face pixels used:", face_mask_bool_analysis_original.sum())

    print("\n--- FINAL CORRECTED METRICS FOR ORIGINAL IMAGE ---")
    print(f"Masked MSE:  {error_mse_analysis_original:.2f}")
    print(f"Masked PSNR: {error_psnr_analysis_original:.2f}")
    print(f"Masked SSIM: {error_ssim_analysis_original:.4f}")



"""### Comparison of Metrics: Inpainted vs. Original"""

import pandas as pd

# Metrics for Inpainted Image (from cell 4ab5f8c9, using manbhav_image.png data)
# NOTE: Re-running that cell would overwrite these variables, so we are assuming they are still from manbhav_image.png's analysis
# The current notebook state has: error_mse_analysis and error_ssim_analysis (from mask.png analysis)
# We need to manually extract the manbhav_image.png metrics again, or re-run the relevant cells with specific paths.

# Let's extract the metrics from the inpainted 'manbhav_image.png' explicitly.
# We will reuse the code from cell 4ab5f8c9, but point to manbhav_image.png

result_image_path_manbhav = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000/manbhav_image.png'
img_bgr_manbhav = cv2.imread(result_image_path_manbhav)

error_mse_analysis_manbhav = None
error_psnr_manbhav = None
error_ssim_analysis_manbhav = None

if img_bgr_manbhav is None:
    print(f"Error: Could not load {result_image_path_manbhav}")
else:
    h_manbhav, w_manbhav, c_manbhav = img_bgr_manbhav.shape
    img_orig_analysis_manbhav = img_bgr_manbhav[:, 0:h_manbhav] # The inpainted input image
    img_render_analysis_manbhav = img_bgr_manbhav[:, h_manbhav:h_manbhav*2] # The textured 3D mesh rendered to 2D

    seg_map_analysis_manbhav = get_segmentation(net, img_orig_analysis_manbhav, device)
    face_mask_bool_analysis_manbhav = make_face_mask(seg_map_analysis_manbhav).astype(bool)

    img_orig_f_analysis_manbhav = img_orig_analysis_manbhav.astype(np.float32)
    img_render_f_analysis_manbhav = img_render_analysis_manbhav.astype(np.float32)

    diff_analysis_manbhav = (img_orig_f_analysis_manbhav - img_render_f_analysis_manbhav) ** 2
    masked_diff_analysis_manbhav = diff_analysis_manbhav[face_mask_bool_analysis_manbhav]
    error_mse_analysis_manbhav = masked_diff_analysis_manbhav.mean()

    mask_for_psnr_manbhav = face_mask_bool_analysis_manbhav.astype(np.uint8) * 255
    temp_orig_for_psnr_manbhav = cv2.bitwise_and(img_orig_analysis_manbhav, img_orig_analysis_manbhav, mask=mask_for_psnr_manbhav)
    temp_render_for_psnr_manbhav = cv2.bitwise_and(img_render_analysis_manbhav, img_render_analysis_manbhav, mask=mask_for_psnr_manbhav)
    error_psnr_manbhav = psnr(temp_orig_for_psnr_manbhav, temp_render_for_psnr_manbhav, data_range=255)

    ys_analysis_manbhav, xs_analysis_manbhav = np.where(face_mask_bool_analysis_manbhav)
    ymin_analysis_manbhav, ymax_analysis_manbhav = ys_analysis_manbhav.min(), ys_analysis_manbhav.max()
    xmin_analysis_manbhav, xmax_analysis_manbhav = xs_analysis_manbhav.min(), xs_analysis_manbhav.max()

    crop_orig_analysis_manbhav = img_orig_analysis_manbhav[ymin_analysis_manbhav:ymax_analysis_manbhav, xmin_analysis_manbhav:xmax_analysis_manbhav]
    crop_render_analysis_manbhav = img_render_analysis_manbhav[ymin_analysis_manbhav:ymax_analysis_manbhav, xmin_analysis_manbhav:xmax_analysis_manbhav]
    error_ssim_analysis_manbhav = ssim(
        crop_orig_analysis_manbhav,
        crop_render_analysis_manbhav,
        channel_axis=-1,
        data_range=255
    )

# Create a DataFrame for comparison
data = {
    'Metric': ['MSE', 'PSNR', 'SSIM'],
    'Inpainted (manbhav_image.jpg)': [f'{error_mse_analysis_manbhav:.2f}', f'{error_psnr_manbhav:.2f}', f'{error_ssim_analysis_manbhav:.4f}'],
    'Original (manbhav_image.jpg)': [f'{error_mse_analysis_original:.2f}', f'{error_psnr_analysis_original:.2f}', f'{error_ssim_analysis_original:.4f}']
}
df_comparison = pd.DataFrame(data)

display(df_comparison)

print("\nInterpretation:\n- Lower MSE and higher PSNR generally indicate better reconstruction quality.\n- Higher SSIM indicates better structural similarity.\n- You can observe how the inpainting process affects these metrics.")

import glob

files = glob.glob('/content/**/results/**/*.obj', recursive=True)
print(files)

import os

folder = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000'
print(os.listdir(folder))

import trimesh

mesh_path = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000/manbhav_image.obj'

mesh = trimesh.load(mesh_path)
mesh.show()

"""### Visualizing Preprocessing Steps for `manbhav_image.png`

This section demonstrates the individual preprocessing steps on the original image:

1.  **Landmark Detection (MTCNN)**: Identifies key facial points.
2.  **Face Parsing (BiSeNet)**: Segmenting the face from the background.
3.  **Binary Occlusion Masking**: Creating a mask for non-face regions, suitable for inpainting or other operations.
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from mtcnn import MTCNN
import os

# Define the path to the original image
original_image_path = '/content/drive/MyDrive/my_images/manbhav_image.jpg' # Corrected from .png to .jpg

# --- 1. Load the original image ---
img_bgr = cv2.imread(original_image_path)
if img_bgr is None:
    print(f"Error: Could not load {original_image_path}")
else:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 4, figsize=(20, 6))

    # --- 2. Landmark Detection (MTCNN) ---
    detector = MTCNN() # Initialize MTCNN (if not already done)
    results = detector.detect_faces(img_rgb)

    img_landmarks = img_rgb.copy()
    if len(results) > 0:
        keypoints = results[0]['keypoints']
        for point in ['left_eye', 'right_eye', 'nose', 'mouth_left', 'mouth_right']:
            cv2.circle(img_landmarks, keypoints[point], 5, (0, 255, 0), -1) # Green circles
        axes[1].set_title('Landmark Detection (MTCNN)')
    else:
        axes[1].set_title('No Face Detected for Landmarks')

    # --- 3. Face Parsing (BiSeNet) ---
    # Ensure BiSeNet model and functions are loaded from previous cells
    # (net, get_segmentation, make_face_mask, device assumed in scope)
    seg_map = get_segmentation(net, img_bgr, device) # Using the helper from cell 01ZM-x1x3mHb and L9XklNUA3wll
    face_mask_bool = make_face_mask(seg_map) # Using the helper from cell 01ZM-x1x3mHb
    face_mask_vis = (face_mask_bool * 255).astype(np.uint8)

    # Apply the face mask to the original image
    img_face_only = cv2.bitwise_and(img_rgb, img_rgb, mask=face_mask_vis)
    axes[2].set_title('Face Parsing (BiSeNet)')

    # --- 4. Binary Occlusion Masking ---
    occlusion_mask_vis = (~face_mask_bool * 255).astype(np.uint8)
    axes[3].set_title('Binary Occlusion Mask')

    # --- 5. Display Results ---
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    axes[1].imshow(img_landmarks)
    axes[1].axis('off')

    axes[2].imshow(img_face_only) # Display the masked face
    axes[2].axis('off')

    axes[3].imshow(occlusion_mask_vis, cmap='gray')
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

"""### Identify Images for New Analysis

Let's first list all image files in your designated input folder, excluding `manbhav_image.jpg`, to select 4 for the new comparison.
"""

import os

INPUT_IMG_FOLDER = '/content/drive/MyDrive/my_images'
image_exts = ('.png', '.jpg', '.jpeg', '.bmp')

# Get all files, filter for images, and exclude 'manbhav_image.jpg' and 'alanritchson.jpeg'
all_image_files = [
    f for f in sorted(os.listdir(INPUT_IMG_FOLDER))
    if f.lower().endswith(image_exts) and f.lower() not in ('manbhav_image.jpg', 'alanritchson.jpeg')
]

print(f"Available images for new analysis (excluding manbhav_image.jpg and alanritchson.jpeg): {len(all_image_files)} files")
for i, img_file in enumerate(all_image_files):
    print(f"  {i+1}. {img_file}")

# Select up to 10 images for the new analysis from the available files
images_for_new_analysis = all_image_files[:10]

if not images_for_new_analysis:
    print("\n⚠️ No new images found for analysis. Please ensure you've uploaded images to '/content/drive/MyDrive/my_images'.")
else:
    print("\n✅ Selected images for this round of analysis:")
    for img_file in images_for_new_analysis:
        print(f"  - {img_file}")

"""### Re-run Face Detection (MTCNN)

Since you've added new images, we need to re-run the MTCNN detection to ensure landmark files are generated for all images in `my_images`.
"""

import os
import cv2
from mtcnn import MTCNN

# 1. Define your image folder
image_folder = '/content/drive/MyDrive/my_images'
detections_folder = os.path.join(image_folder, 'detections')
os.makedirs(detections_folder, exist_ok=True)

# 2. Initialize MTCNN detector
detector = MTCNN()

print("Generating MTCNN detections for all images in '/content/drive/MyDrive/my_images'...")

# 3. Process each image in the folder
for filename in os.listdir(image_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(image_folder, filename)

        # Read image
        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read image: {filename}, skipping.")
            continue

        # MTCNN requires RGB images
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Detect face(s)
        results = detector.detect_faces(img_rgb)

        if len(results) > 0:
            # We take the first (most prominent) face detected
            keypoints = results[0]['keypoints']

            # Deep3DFaceRecon expects 5 landmarks in this exact order:
            # Left Eye, Right Eye, Nose, Mouth Left, Mouth Right
            landmarks = [
                keypoints['left_eye'],
                keypoints['right_eye'],
                keypoints['nose'],
                keypoints['mouth_left'],
                keypoints['mouth_right']
            ]

            # Create a .txt file with the exact same name as the image
            txt_filename = os.path.splitext(filename)[0] + '.txt'
            txt_path = os.path.join(detections_folder, txt_filename)

            # Write coordinates to the text file
            with open(txt_path, 'w') as f:
                for pt in landmarks:
                    f.write(f"{pt[0]} {pt[1]}\n")

            print(f"  ✅ Saved landmarks for {filename}")
        else:
            print(f"  ❌ No face detected in {filename}")

print("\n✅ MTCNN detection complete for all images.")

"""### Re-run Inpainting Preprocessing

This step will re-process all images in `my_images` (including any newly added ones) by generating inpainted versions using BiSeNet and saving them to the `input_images_inpainted` folder. This ensures all new images are ready for 3D reconstruction with occlusion handling.
"""

# ===== OCCLUSION MODULE START (Re-run for all images) =====
import os, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
import torchvision.transforms as transforms

RECON_ROOT = '/content/Deep3DFaceRecon_pytorch'
INPUT_IMG_FOLDER = '/content/drive/MyDrive/my_images'
INPAINTED_FOLDER_REL = 'input_images_inpainted'
INPAINTED_IMG_FOLDER = os.path.join(RECON_ROOT, INPAINTED_FOLDER_REL)

# Keep relative paths consistent with the later `!python test.py ...` cell.
os.makedirs(INPAINTED_IMG_FOLDER, exist_ok=True)
os.makedirs(os.path.join(INPAINTED_IMG_FOLDER, 'detections'), exist_ok=True)
os.chdir(RECON_ROOT)

# --- Explicitly redefine BiSeNet helpers here for robustness against stale kernel state ---
# These are copied from cell 01ZM-x1x3mHb and L9XklNUA3wll to ensure correct definitions are used.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bisenet_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
FACE_LABELS = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

# Load BiSeNet if not already loaded (re-checking)
if 'net' not in locals() or not isinstance(net, BiSeNet):
    sys.path.insert(0, '/content/Deep3DFaceRecon_pytorch/face-parsing.PyTorch')
    from model import BiSeNet
    weights_path = '/content/Deep3DFaceRecon_pytorch/face-parsing.PyTorch/res/cp/79999_iter.pth'
    net = BiSeNet(n_classes=19)
    state = torch.load(weights_path, map_location=device)
    if all(k.startswith('module.') for k in state.keys()):
        state = {k[7:]: v for k, v in state.items()}
    net.load_state_dict(state)
    net.to(device)
    net.eval()
    print('✓ BiSeNet loaded for inpainting!')

@torch.no_grad()
def get_segmentation(net, img_bgr, device):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inp = bisenet_transform(pil).unsqueeze(0).to(device)

    out = net(inp)[0]
    seg = out.squeeze(0).argmax(0).cpu().numpy().astype(np.uint8)

    seg = cv2.resize(seg, (img_bgr.shape[1], img_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    return seg

def make_face_mask(seg_map):
    mask = np.zeros_like(seg_map, dtype=np.uint8)

    for lbl in FACE_LABELS:
        mask[seg_map == lbl] = 1

    # Morphological smoothing
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

def inpaint_face(image_bgr, occlusion_mask_uint8_0_255):
    return cv2.inpaint(image_bgr, occlusion_mask_uint8_0_255, 3, cv2.INPAINT_TELEA)

# --- Preprocess ALL input images (segmentation -> occlusion mask -> inpainting) ---
image_exts = ('.png', '.jpg', '.jpeg', '.bmp')
image_files_to_process = [f for f in sorted(os.listdir(INPUT_IMG_FOLDER)) if f.lower().endswith(image_exts)]

# Clear previous inpainted images, but keep the directory structure
shutil.rmtree(INPAINTED_IMG_FOLDER, ignore_errors=True)
os.makedirs(INPAINTED_IMG_FOLDER, exist_ok=True)
os.makedirs(os.path.join(INPAINTED_IMG_FOLDER, 'detections'), exist_ok=True)

# Copy landmark txts so Deep3DFaceRecon keeps the same correspondence.
src_det_dir = os.path.join(INPUT_IMG_FOLDER, 'detections')
dst_det_dir = os.path.join(INPAINTED_IMG_FOLDER, 'detections')
if os.path.exists(src_det_dir):
    for fn in os.listdir(src_det_dir):
        if fn.lower().endswith('.txt'):
            shutil.copy(os.path.join(src_det_dir, fn), os.path.join(dst_det_dir, fn))

print("Starting inpainting for all images...")
for idx, filename in enumerate(image_files_to_process):
    img_path = os.path.join(INPUT_IMG_FOLDER, filename)
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f'⚠️ Skipping unreadable image: {filename}')
        continue

    # Step 1: Face segmentation -> face mask
    seg_map = get_segmentation(net, img_bgr, device)
    face_mask = make_face_mask(seg_map).astype(bool)

    # Step 2: Occlusion mask = NOT(face_mask)
    occlusion_mask = (~face_mask).astype(np.uint8) * 255  # 0 or 255

    # slight smoothing (morphology) to reduce speckles
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    occlusion_mask = cv2.morphologyEx(occlusion_mask, cv2.MORPH_OPEN, kernel)
    occlusion_mask = cv2.morphologyEx(occlusion_mask, cv2.MORPH_CLOSE, kernel)

    # Step 3: Inpainting
    img_inpainted_bgr = inpaint_face(img_bgr, occlusion_mask)

    # Save inpainted image with the SAME filename
    out_path = os.path.join(INPAINTED_IMG_FOLDER, filename)
    cv2.imwrite(out_path, img_inpainted_bgr)
    print(f"  ✅ Inpainted and saved: {filename}")

    # Visualization only if it's one of the selected for new analysis and not manbhav
    if filename in images_for_new_analysis:
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        inpaint_rgb = cv2.cvtColor(img_inpainted_bgr, cv2.COLOR_BGR2RGB)

        face_mask_vis = (face_mask.astype(np.uint8) * 255)
        occlusion_mask_vis = occlusion_mask

        plt.figure(figsize=(16, 4))
        plt.suptitle(f'Inpainting Process for {filename}', fontsize=16)
        plt.subplot(1, 4, 1)
        plt.title('Original')
        plt.imshow(img_rgb)
        plt.axis('off')

        plt.subplot(1, 4, 2)
        plt.title('Face mask')
        plt.imshow(face_mask_vis, cmap='gray')
        plt.axis('off')

        plt.subplot(1, 4, 3)
        plt.title('Occlusion mask')
        plt.imshow(occlusion_mask_vis, cmap='gray')
        plt.axis('off')

        plt.subplot(1, 4, 4)
        plt.title('Inpainted')
        plt.imshow(inpaint_rgb)
        plt.axis('off')
        plt.show()

print('\n✅ Inpainting preprocessing complete. All inpainted images are ready.')
# ===== OCCLUSION MODULE END =====

"""### Re-run Deep3DFaceRecon on Inpainted Images

Now, we'll run the 3D reconstruction model on the *inpainted* images. This will generate 3D meshes and 2D renders for all processed images, including the newly added ones.
"""

# Commented out IPython magic to ensure Python compatibility.
# Change directory to Deep3DFaceRecon_pytorch for test.py command
# %cd /content/Deep3DFaceRecon_pytorch

import os
import shutil

# Assuming images_for_new_analysis is available from previous cells
# And INPAINTED_IMG_FOLDER is defined as '/content/Deep3DFaceRecon_pytorch/input_images_inpainted'

# Create a base directory for individual image results to avoid conflicts
INDIVIDUAL_INPAINTED_RESULTS_BASE = os.path.join(RECON_ROOT, 'individual_inpainted_results')
os.makedirs(INDIVIDUAL_INPAINTED_RESULTS_BASE, exist_ok=True)

# Define paths for checkpoint copying
ORIGINAL_CHECKPOINT_DIR = os.path.join(RECON_ROOT, 'checkpoints', 'face_recon_feat0.2_augment')
INPAINTED_CHECKPOINT_DIR = os.path.join(RECON_ROOT, 'checkpoints', 'face_recon_feat0.2_augment_inpainted')

# Ensure the new checkpoint directory exists for inpainted runs
os.makedirs(INPAINTED_CHECKPOINT_DIR, exist_ok=True)

# Copy the model checkpoint and training options to the new directory
shutil.copy(os.path.join(ORIGINAL_CHECKPOINT_DIR, 'epoch_20.pth'), INPAINTED_CHECKPOINT_DIR)
shutil.copy(os.path.join(ORIGINAL_CHECKPOINT_DIR, 'train_opt.txt'), INPAINTED_CHECKPOINT_DIR)
print(f"✅ Copied checkpoint and options to {INPAINTED_CHECKPOINT_DIR}")

# Clean up previous Deep3DFaceRecon results for these images to avoid conflicts
print("Cleaning up previous Deep3DFaceRecon output directories for inpainted images...")
for img_filename in images_for_new_analysis:
    base_name = os.path.splitext(img_filename)[0]
    # Clean up results created with the specific name used for inpainted results
    output_result_dir_specific = os.path.join(
        RECON_ROOT,
        'checkpoints',
        'face_recon_feat0.2_augment_inpainted',
        'results',
        base_name
    )
    if os.path.exists(output_result_dir_specific):
        shutil.rmtree(output_result_dir_specific)
        print(f"  Removed specific inpainted results: {output_result_dir_specific}")

    # Also clean up any generic results that might have been created from previous runs
    output_result_dir_generic = os.path.join(
        RECON_ROOT,
        'checkpoints',
        'face_recon_feat0.2_augment',
        'results',
        base_name
    )
    if os.path.exists(output_result_dir_generic):
        shutil.rmtree(output_result_dir_generic)
        print(f"  Removed generic results: {output_result_dir_generic}")

print("Cleanup complete.")

print('\nStarting Deep3DFaceRecon for each inpainted image individually...')

for img_filename in images_for_new_analysis:
    print(f"  Processing inpainted: {img_filename}")

    # Create a unique temporary folder for this image's input and detections
    temp_input_folder_name = os.path.splitext(img_filename)[0]
    temp_input_path = os.path.join(INDIVIDUAL_INPAINTED_RESULTS_BASE, temp_input_folder_name)
    os.makedirs(temp_input_path, exist_ok=True)
    os.makedirs(os.path.join(temp_input_path, 'detections'), exist_ok=True)

    # Copy the inpainted image and its detection file to the temporary folder
    src_img_path = os.path.join(INPAINTED_IMG_FOLDER, img_filename)
    dst_img_path = os.path.join(temp_input_path, img_filename)
    shutil.copy(src_img_path, dst_img_path)

    src_det_path = os.path.join(INPAINTED_IMG_FOLDER, 'detections', os.path.splitext(img_filename)[0] + '.txt')
    dst_det_path = os.path.join(temp_input_path, 'detections', os.path.splitext(img_filename)[0] + '.txt')
    if os.path.exists(src_det_path):
        shutil.copy(src_det_path, dst_det_path)
    else:
        print(f"    ⚠️ Warning: Detection file not found for {img_filename} at {src_det_path}")

    # Run test.py for this single image, using a unique name for inpainted results
    !python test.py \
      --name=face_recon_feat0.2_augment_inpainted \
      --epoch=20 \
      --img_folder={temp_input_path} \
      --dataset_mode= \
      --phase=test

    print(f"  ✅ Deep3DFaceRecon completed for inpainted {img_filename}")

print('\n✅ Deep3DFaceRecon complete for all selected inpainted images.')

"""### Prepare and Re-run Deep3DFaceRecon on Original Images

To compare, we need to run the reconstruction process again, but this time using the *original* input images (without any inpainting). We'll update the temporary folder for these images and their detection files, then run the model.
"""

# Commented out IPython magic to ensure Python compatibility.
import os, shutil

# Define paths
RECON_ROOT = '/content/Deep3DFaceRecon_pytorch'
INPUT_IMG_FOLDER = '/content/drive/MyDrive/my_images'
TEMP_ORIG_INPUT_FOLDER_NAME = 'original_images_for_recon'
TEMP_ORIG_INPUT_FOLDER = os.path.join(RECON_ROOT, TEMP_ORIG_INPUT_FOLDER_NAME)
TEMP_ORIG_DETECTIONS_FOLDER = os.path.join(TEMP_ORIG_INPUT_FOLDER, 'detections')

# Clean previous original images, but keep the directory structure
shutil.rmtree(TEMP_ORIG_INPUT_FOLDER, ignore_errors=True)
os.makedirs(TEMP_ORIG_INPUT_FOLDER, exist_ok=True)
os.makedirs(TEMP_ORIG_DETECTIONS_FOLDER, exist_ok=True)

# Copy original images to the temporary folder
image_exts = ('.png', '.jpg', '.jpeg', '.bmp')
for filename in os.listdir(INPUT_IMG_FOLDER):
    if filename.lower().endswith(image_exts):
        shutil.copy(os.path.join(INPUT_IMG_FOLDER, filename), os.path.join(TEMP_ORIG_INPUT_FOLDER, filename))

# Copy landmark detection files to the temporary detections folder
s_det_dir = os.path.join(INPUT_IMG_FOLDER, 'detections')
if os.path.exists(s_det_dir):
    for fn in os.listdir(s_det_dir):
        if fn.lower().endswith('.txt'):
            shutil.copy(os.path.join(s_det_dir, fn), os.path.join(TEMP_ORIG_DETECTIONS_FOLDER, fn))

print(f'✅ Copied all original images to {TEMP_ORIG_INPUT_FOLDER}')
print(f'✅ Copied all landmark detections to {TEMP_ORIG_DETECTIONS_FOLDER}')

# Change directory to Deep3DFaceRecon_pytorch for test.py command
# %cd /content/Deep3DFaceRecon_pytorch

# Commented out IPython magic to ensure Python compatibility.
import os
import shutil

# Change directory to Deep3DFaceRecon_pytorch for test.py command
# %cd /content/Deep3DFaceRecon_pytorch

# Assuming images_for_new_analysis is available from previous cells
# And TEMP_ORIG_INPUT_FOLDER is defined as '/content/Deep3DFaceRecon_pytorch/original_images_for_recon'

# Create a base directory for individual image results to avoid conflicts
INDIVIDUAL_ORIGINAL_RESULTS_BASE = os.path.join(RECON_ROOT, 'individual_original_results')
os.makedirs(INDIVIDUAL_ORIGINAL_RESULTS_BASE, exist_ok=True)

# Define paths for checkpoint copying
ORIGINAL_CHECKPOINT_DIR = os.path.join(RECON_ROOT, 'checkpoints', 'face_recon_feat0.2_augment')
ORIGINAL_RESULTS_CHECKPOINT_DIR = os.path.join(RECON_ROOT, 'checkpoints', 'face_recon_feat0.2_augment_original')

# Ensure the new checkpoint directory exists for original runs
os.makedirs(ORIGINAL_RESULTS_CHECKPOINT_DIR, exist_ok=True)

# Copy the model checkpoint and training options to the new directory
shutil.copy(os.path.join(ORIGINAL_CHECKPOINT_DIR, 'epoch_20.pth'), ORIGINAL_RESULTS_CHECKPOINT_DIR)
shutil.copy(os.path.join(ORIGINAL_CHECKPOINT_DIR, 'train_opt.txt'), ORIGINAL_RESULTS_CHECKPOINT_DIR)
print(f"✅ Copied checkpoint and options to {ORIGINAL_RESULTS_CHECKPOINT_DIR}")

# Clean up previous Deep3DFaceRecon results for these images to avoid conflicts
print("Cleaning up previous Deep3DFaceRecon output directories for original images...")
for img_filename in images_for_new_analysis:
    base_name = os.path.splitext(img_filename)[0]
    # Clean up results created with the specific name used for original results
    output_result_dir_specific = os.path.join(
        RECON_ROOT,
        'checkpoints',
        'face_recon_feat0.2_augment_original',
        'results',
        base_name
    )
    if os.path.exists(output_result_dir_specific):
        shutil.rmtree(output_result_dir_specific)
        print(f"  Removed specific original results: {output_result_dir_specific}")

    # Also clean up any generic results that might have been created from previous runs
    output_result_dir_generic = os.path.join(
        RECON_ROOT,
        'checkpoints',
        'face_recon_feat0.2_augment',
        'results',
        base_name
    )
    if os.path.exists(output_result_dir_generic):
        shutil.rmtree(output_result_dir_generic)
        print(f"  Removed generic results: {output_result_dir_generic}")

print("Cleanup complete.")

print('\nStarting Deep3DFaceRecon for each original image individually...')

for img_filename in images_for_new_analysis:
    print(f"  Processing original: {img_filename}")

    # Create a unique temporary folder for this image's input and detections
    temp_input_folder_name = os.path.splitext(img_filename)[0]
    temp_input_path = os.path.join(INDIVIDUAL_ORIGINAL_RESULTS_BASE, temp_input_folder_name)
    os.makedirs(temp_input_path, exist_ok=True)
    os.makedirs(os.path.join(temp_input_path, 'detections'), exist_ok=True)

    # Copy the original image and its detection file to the temporary folder
    src_img_path = os.path.join(INPUT_IMG_FOLDER, img_filename)
    dst_img_path = os.path.join(temp_input_path, img_filename)
    shutil.copy(src_img_path, dst_img_path)

    src_det_path = os.path.join(INPUT_IMG_FOLDER, 'detections', os.path.splitext(img_filename)[0] + '.txt')
    dst_det_path = os.path.join(temp_input_path, 'detections', os.path.splitext(img_filename)[0] + '.txt')
    if os.path.exists(src_det_path):
        shutil.copy(src_det_path, dst_det_path)
    else:
        print(f"    ⚠️ Warning: Detection file not found for {img_filename} at {src_det_path}")

    # Run test.py for this single image, using a unique name for original results
    !python test.py \
      --name=face_recon_feat0.2_augment_original \
      --epoch=20 \
      --img_folder={temp_input_path} \
      --dataset_mode= \
      --phase=test

    print(f"  ✅ Deep3DFaceRecon completed for original {img_filename}")

print('\n✅ Deep3DFaceRecon complete for all selected original images.')



"""### Metric Analysis and Comparison for Selected Images

Now, let's perform the metric analysis (MSE, PSNR, SSIM) for the 4 selected images, comparing the results of the *inpainted* reconstruction versus the *original* reconstruction. This will highlight the impact of the inpainting step.
"""

# Commented out IPython magic to ensure Python compatibility.
# Change directory
# %cd /content/Deep3DFaceRecon_pytorch

import os
import shutil
import cv2

# -------------------------------
# SETUP
# -------------------------------
INDIVIDUAL_ORIGINAL_RESULTS_BASE = os.path.join(RECON_ROOT, 'individual_original_results')
os.makedirs(INDIVIDUAL_ORIGINAL_RESULTS_BASE, exist_ok=True)

ORIGINAL_CHECKPOINT_DIR = os.path.join(RECON_ROOT, 'checkpoints', 'face_recon_feat0.2_augment')
ORIGINAL_RESULTS_CHECKPOINT_DIR = os.path.join(RECON_ROOT, 'checkpoints', 'face_recon_feat0.2_augment_original')

os.makedirs(ORIGINAL_RESULTS_CHECKPOINT_DIR, exist_ok=True)

# Copy checkpoint safely (no overwrite spam)
for file in ['epoch_20.pth', 'train_opt.txt']:
    src = os.path.join(ORIGINAL_CHECKPOINT_DIR, file)
    dst = os.path.join(ORIGINAL_RESULTS_CHECKPOINT_DIR, file)
    if not os.path.exists(dst):
        shutil.copy(src, dst)

print(f"✅ Checkpoint ready at {ORIGINAL_RESULTS_CHECKPOINT_DIR}")

# -------------------------------
# CLEAN OLD RESULTS
# -------------------------------
print("Cleaning previous original results...")

for img_filename in images_for_new_analysis:
    base = os.path.splitext(img_filename)[0]

    for folder in [
        'face_recon_feat0.2_augment_original',
        'face_recon_feat0.2_augment'
    ]:
        path = os.path.join(RECON_ROOT, 'checkpoints', folder, 'results', base)
        if os.path.exists(path):
            shutil.rmtree(path)
            print(f"Removed: {path}")

print("Cleanup complete.\n")

# -------------------------------
# MAIN LOOP
# -------------------------------
print("Starting ORIGINAL reconstruction...\n")

for img_filename in images_for_new_analysis:

    print(f"Processing original: {img_filename}")

    base_name = os.path.splitext(img_filename)[0]

    temp_input_path = os.path.join(INDIVIDUAL_ORIGINAL_RESULTS_BASE, base_name)
    os.makedirs(temp_input_path, exist_ok=True)
    os.makedirs(os.path.join(temp_input_path, 'detections'), exist_ok=True)

    # -------------------------------
    # 🔥 FIX 1: HANDLE IMAGE FORMAT
    # -------------------------------
    src_img_path = os.path.join(INPUT_IMG_FOLDER, img_filename)

    if not os.path.exists(src_img_path):
        print(f"❌ Missing original image: {src_img_path}")
        continue

    img = cv2.imread(src_img_path)

    if img is None:
        print(f"❌ Failed to read original image: {img_filename}")
        continue

    # 🔥 ALWAYS convert to .jpg
    new_img_name = base_name + ".jpg"
    dst_img_path = os.path.join(temp_input_path, new_img_name)
    cv2.imwrite(dst_img_path, img)

    # -------------------------------
    # 🔥 FIX 2: DETECTION FILE
    # -------------------------------
    src_det_path = os.path.join(INPUT_IMG_FOLDER, 'detections', base_name + '.txt')
    dst_det_path = os.path.join(temp_input_path, 'detections', base_name + '.txt')

    if os.path.exists(src_det_path):
        shutil.copy(src_det_path, dst_det_path)
    else:
        print(f"⚠️ No detection file for {img_filename}")

    # -------------------------------
    # RUN MODEL
    # -------------------------------
    !python test.py \
      --name=face_recon_feat0.2_augment_original \
      --epoch=20 \
      --img_folder={temp_input_path} \
      --dataset_mode= \
      --phase=test

    # -------------------------------
    # 🔥 FIX 3: VERIFY OUTPUT
    # -------------------------------
    result_path = os.path.join(
        RECON_ROOT,
        'checkpoints',
        'face_recon_feat0.2_augment_original',
        'results',
        base_name,
        'epoch_20_000000',
        base_name + '.png'
    )

    if os.path.exists(result_path):
        print(f"✅ Success: {base_name}.png generated")
    else:
        print(f"❌ ERROR: Output NOT generated for {img_filename}")

print("\n✅ ORIGINAL reconstruction completed.")

import trimesh

mesh_path = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment_inpainted/results/000005/epoch_20_000000/000005.obj'

mesh = trimesh.load(mesh_path)
mesh.show()

import trimesh

mesh_path = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment_inpainted/results/000002/epoch_20_000000/000002.obj'

mesh = trimesh.load(mesh_path)
mesh.show()

import trimesh

mesh_path = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment_inpainted/results/000007/epoch_20_000000/000007.obj'

mesh = trimesh.load(mesh_path)
mesh.show()

import trimesh

mesh_path = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment_inpainted/results/henry/epoch_20_000000/henry.obj'

mesh = trimesh.load(mesh_path)
mesh.show()

import trimesh

mesh_path = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment_inpainted/results/000024/epoch_20_000000/000024.obj'

mesh = trimesh.load(mesh_path)
mesh.show()

"""### Identity Similarity (ArcFace Cosine Similarity)

We will use the ArcFace model from the `insightface` library (which was cloned during setup) to extract face embeddings. We'll then compute the cosine similarity between the embeddings of the original and rendered images to quantify how well the identity is preserved after 3D reconstruction.

Now, let's re-run the metric analysis to include Identity Similarity.
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import pandas as pd
import os
from sklearn.metrics.pairwise import cosine_similarity

import insightface
from insightface.app import FaceAnalysis

# -------------------------------
# ArcFace Setup
# -------------------------------
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(1024, 1024))

# -------------------------------
# Utility
# -------------------------------
def add_padding(img, pad=80):
    return cv2.copyMakeBorder(img, pad, pad, pad, pad,
                              cv2.BORDER_CONSTANT, value=[0,0,0])

def center_crop(img, size=224):
    h, w = img.shape[:2]
    cx, cy = w//2, h//2
    half = size//2
    return img[max(0, cy-half):cy+half, max(0, cx-half):cx+half]

# -------------------------------
# Embedding
# -------------------------------
def get_face_embedding(image_rgb, filename, tag):
    img = image_rgb.copy()
    faces = app.get(img)

    if len(faces) > 0:
        faces = sorted(faces, key=lambda x: (x.bbox[2]-x.bbox[0])*(x.bbox[3]-x.bbox[1]), reverse=True)
        return faces[0].embedding
    else:
        print(f"⚠️ Fallback crop used ({filename}, {tag})")
        crop = center_crop(img, 224)
        crop = cv2.resize(crop, (224, 224))
        faces = app.get(crop)
        return faces[0].embedding if len(faces) > 0 else None

# -------------------------------
# MAIN LOOP
# -------------------------------
all_data = []

for image_filename in images_for_new_analysis:

    print(f"\n--- {image_filename} ---")

    base_name = os.path.splitext(image_filename)[0]

    # 🔥 FIX: robust filename handling
    output_name = base_name + ".png"

    inp_path = os.path.join(
        RECON_ROOT, 'checkpoints',
        'face_recon_feat0.2_augment_inpainted',
        'results',
        base_name,
        'epoch_20_000000',
        output_name
    )

    orig_path = os.path.join(
        RECON_ROOT, 'checkpoints',
        'face_recon_feat0.2_augment_original',
        'results',
        base_name,
        'epoch_20_000000',
        output_name
    )

    # -------------------------------
    # LOAD INPAINTED
    # -------------------------------
    img = cv2.imread(inp_path)
    if img is None:
        print(f"❌ Missing inpainted output: {inp_path}")
        continue

    h, w, _ = img.shape
    img_orig = img[:, :w//2]
    img_render = img[:, w//2:]

    seg = get_segmentation(net, img_orig, device)
    mask = make_face_mask(seg).astype(bool)

    diff = (img_orig.astype(np.float32) - img_render.astype(np.float32))**2
    mse_inp = diff[mask].mean()

    mask_u = mask.astype(np.uint8)*255

    psnr_inp = psnr(
        cv2.bitwise_and(img_orig, img_orig, mask=mask_u),
        cv2.bitwise_and(img_render, img_render, mask=mask_u),
        data_range=255
    )

    ys, xs = np.where(mask)
    crop_o = img_orig[ys.min():ys.max(), xs.min():xs.max()]
    crop_r = img_render[ys.min():ys.max(), xs.min():xs.max()]

    ssim_inp = ssim(crop_o, crop_r, channel_axis=-1, data_range=255)

    # -------------------------------
    # IDENTITY
    # -------------------------------
    rgb_orig = add_padding(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
    rgb_render = add_padding(cv2.cvtColor(img_render, cv2.COLOR_BGR2RGB))

    emb1 = get_face_embedding(rgb_orig, image_filename, "Inp-Orig")
    emb2 = get_face_embedding(rgb_render, image_filename, "Inp-Render")

    sim_inp = cosine_similarity(emb1.reshape(1,-1), emb2.reshape(1,-1))[0][0] if emb1 is not None and emb2 is not None else np.nan

    # -------------------------------
    # LOAD ORIGINAL
    # -------------------------------
    img = cv2.imread(orig_path)
    if img is None:
        print(f"❌ Missing original output: {orig_path}")
        continue

    h, w, _ = img.shape
    img_orig = img[:, :w//2]
    img_render = img[:, w//2:]

    seg = get_segmentation(net, img_orig, device)
    mask = make_face_mask(seg).astype(bool)

    diff = (img_orig.astype(np.float32) - img_render.astype(np.float32))**2
    mse_orig = diff[mask].mean()

    mask_u = mask.astype(np.uint8)*255

    psnr_orig = psnr(
        cv2.bitwise_and(img_orig, img_orig, mask=mask_u),
        cv2.bitwise_and(img_render, img_render, mask=mask_u),
        data_range=255
    )

    ys, xs = np.where(mask)
    crop_o = img_orig[ys.min():ys.max(), xs.min():xs.max()]
    crop_r = img_render[ys.min():ys.max(), xs.min():xs.max()]

    ssim_orig = ssim(crop_o, crop_r, channel_axis=-1, data_range=255)

    rgb_orig = add_padding(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
    rgb_render = add_padding(cv2.cvtColor(img_render, cv2.COLOR_BGR2RGB))

    emb1 = get_face_embedding(rgb_orig, image_filename, "Orig-Orig")
    emb2 = get_face_embedding(rgb_render, image_filename, "Orig-Render")

    sim_orig = cosine_similarity(emb1.reshape(1,-1), emb2.reshape(1,-1))[0][0] if emb1 is not None and emb2 is not None else np.nan

    # -------------------------------
    # PRINT
    # -------------------------------
    print(f"MSE: {mse_inp:.2f} | {mse_orig:.2f}")
    print(f"PSNR: {psnr_inp:.2f} | {psnr_orig:.2f}")
    print(f"SSIM: {ssim_inp:.3f} | {ssim_orig:.3f}")
    print(f"ID: {sim_inp:.3f} | {sim_orig:.3f}")

    all_data.append({
        "Image": image_filename,
        "Inpainted_MSE": mse_inp,
        "Original_MSE": mse_orig,
        "Inpainted_PSNR": psnr_inp,
        "Original_PSNR": psnr_orig,
        "Inpainted_SSIM": ssim_inp,
        "Original_SSIM": ssim_orig,
        "Inpainted_ID": sim_inp,
        "Original_ID": sim_orig
    })

# -------------------------------
# FINAL
# -------------------------------
df = pd.DataFrame(all_data)
print("\n=== FINAL RESULTS ===")
display(df)

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

RECON_ROOT = '/content/Deep3DFaceRecon_pytorch'

ORIG_ROOT = os.path.join(
    RECON_ROOT,
    'checkpoints',
    'face_recon_feat0.2_augment_original',
    'results'
)

INP_ROOT = os.path.join(
    RECON_ROOT,
    'checkpoints',
    'face_recon_feat0.2_augment_inpainted',
    'results'
)

for base_name in os.listdir(ORIG_ROOT):

    orig_path = os.path.join(ORIG_ROOT, base_name, 'epoch_20_000000', base_name + '.png')
    inp_path  = os.path.join(INP_ROOT, base_name, 'epoch_20_000000', base_name + '.png')

    if not os.path.exists(orig_path) or not os.path.exists(inp_path):
        continue

    print(f"Processing: {base_name}")

    img_orig_full = cv2.imread(orig_path)
    img_inp_full  = cv2.imread(inp_path)

    if img_orig_full is None or img_inp_full is None:
        continue

    h = img_orig_full.shape[0]

    # Split
    orig_input  = img_orig_full[:, 0:h]
    orig_render = img_orig_full[:, h:2*h]

    inp_input   = img_inp_full[:, 0:h]
    inp_render  = img_inp_full[:, h:2*h]

    # Mask (use common mask for fairness)
    gray1 = cv2.cvtColor(orig_render, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(inp_render,  cv2.COLOR_BGR2GRAY)

    _, mask1 = cv2.threshold(gray1, 5, 255, cv2.THRESH_BINARY)
    _, mask2 = cv2.threshold(gray2, 5, 255, cv2.THRESH_BINARY)

    mask = cv2.bitwise_and(mask1, mask2)

    def heatmap(inp, rend):
        inp = cv2.bitwise_and(inp, inp, mask=mask)
        rend = cv2.bitwise_and(rend, rend, mask=mask)

        diff = np.mean(np.abs(inp.astype(np.float32) - rend.astype(np.float32)), axis=2)
        diff = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        hm = cv2.applyColorMap(diff, cv2.COLORMAP_JET)
        hm = cv2.bitwise_and(hm, hm, mask=mask)

        return hm

    hm_orig = heatmap(orig_input, orig_render)
    hm_inp  = heatmap(inp_input, inp_render)

    # Display
    plt.figure(figsize=(16,5))

    plt.subplot(1,5,1)
    plt.imshow(cv2.cvtColor(orig_input, cv2.COLOR_BGR2RGB))
    plt.title("Input")
    plt.axis('off')

    plt.subplot(1,5,2)
    plt.imshow(cv2.cvtColor(orig_render, cv2.COLOR_BGR2RGB))
    plt.title("Render (Original)")
    plt.axis('off')

    plt.subplot(1,5,3)
    plt.imshow(cv2.cvtColor(hm_orig, cv2.COLOR_BGR2RGB))
    plt.title("Heatmap (Original)")
    plt.axis('off')

    plt.subplot(1,5,4)
    plt.imshow(cv2.cvtColor(inp_render, cv2.COLOR_BGR2RGB))
    plt.title("Render (Inpainted)")
    plt.axis('off')

    plt.subplot(1,5,5)
    plt.imshow(cv2.cvtColor(hm_inp, cv2.COLOR_BGR2RGB))
    plt.title("Heatmap (Inpainted)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

import cv2
import numpy as np
import os
import pandas as pd
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr

# --- 1. PATHS ---
ORIG_DIR = '/content/drive/MyDrive/my_images/'
MP_DIR   = '/content/drive/MyDrive/MediaPipe_Results/'
D3D_DIR  = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/original_images_for_recon/epoch_20_000000/'
INP_DIR  = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000/'

results = []
image_files = [f for f in os.listdir(ORIG_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Calculating Metrics for {len(image_files)} images...")

for name in image_files:
    base_name = os.path.splitext(name)[0]
    orig = cv2.imread(os.path.join(ORIG_DIR, name))
    if orig is None: continue
    h, w, _ = orig.shape

    def add_to_results(model_name, folder_path, is_strip=True):
        ext = os.path.splitext(name)[1] if model_name == 'MediaPipe' else '.png'
        img_path = os.path.join(folder_path, base_name + ext)
        img = cv2.imread(img_path)

        if img is not None:
            if is_strip and img.shape[1] >= w * 2:
                img = img[:, w:2*w]

            img = cv2.resize(img, (w, h))

            # --- CALCULATIONS ---
            # 1. MSE
            mse_val = np.mean((orig.astype(np.float32) - img.astype(np.float32)) ** 2)

            # 2. RMSE (Root Mean Square Error)
            rmse_val = np.sqrt(mse_val)

            # 3. Normalized RMSE (Error as a fraction of total pixel range 0-255)
            n_rmse = rmse_val / 255.0

            # 4. SSIM & PSNR
            ssim_val = ssim(orig, img, channel_axis=-1, data_range=255)
            psnr_val = psnr(orig, img, data_range=255)

            results.append({
                'Model': model_name,
                'MSE': mse_val,
                'RMSE': rmse_val,
                'NRMSE': n_rmse,
                'SSIM': ssim_val,
                'PSNR': psnr_val
            })

    add_to_results('MediaPipe', MP_DIR, is_strip=False)
    add_to_results('Deep3D_Standard', D3D_DIR, is_strip=True)
    add_to_results('Custom_Inpainted', INP_DIR, is_strip=True)

# --- 2. SUMMARY TABLE ---
df = pd.DataFrame(results)
if not df.empty:
    # Grouping by Model and taking the mean of all metrics
    final_table = df.groupby('Model')[['MSE', 'RMSE', 'NRMSE', 'PSNR', 'SSIM']].mean()

    # Sorting by SSIM (usually the most respected metric for reconstruction)
    final_table = final_table.sort_values(by='SSIM', ascending=False)

    print("\n" + "="*80)
    print("                UPDATED RESEARCH BENCHMARK (WITH NORMALIZATION)")
    print("="*80)
    print(final_table)
    print("="*80)
    print("Note: NRMSE represents the average pixel error percentage (0 to 1 scale).")

import trimesh
import numpy as np
from scipy.io import loadmat
import os
import pandas as pd

# 1. Define Paths (Based on your folder structure)
MY_INPAINTED_RESULTS = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000'
DATASET_REF_RESULTS = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000'

def evaluate_3d_accuracy(my_obj, ref_obj, my_mat, ref_mat):
    results = {}

    # --- A. Vertex-to-Vertex Distance (Mesh Accuracy) ---
    if os.path.exists(my_obj) and os.path.exists(ref_obj):
        my_mesh = trimesh.load(my_obj)
        ref_mesh = trimesh.load(ref_obj)

        # Deep3D uses a fixed topology (same number of vertices)
        if len(my_mesh.vertices) == len(ref_mesh.vertices):
            dist = np.linalg.norm(my_mesh.vertices - ref_mesh.vertices, axis=1)
            results['3D_Vertex_RMSE'] = np.sqrt(np.mean(dist**2))
            results['3D_Vertex_MAE'] = np.mean(dist)

    # --- B. 3DMM Coefficient Comparison (Shape/Exp Accuracy) ---
    if os.path.exists(my_mat) and os.path.exists(ref_mat):
        my_data, ref_data = loadmat(my_mat), loadmat(ref_mat)
        # Check Cosine Similarity for Identity (id) and Expression (exp)
        for key in ['id', 'exp']:
            if key in my_data and key in ref_data:
                v1, v2 = my_data[key].flatten(), ref_data[key].flatten()
                sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                results[f'{key}_Cosine_Sim'] = sim

    return results

# 2. Batch Compare matched files
comparison_data = []
for file in os.listdir(MY_INPAINTED_RESULTS):
    if file.endswith('.obj'):
        ref_path = os.path.join(DATASET_REF_RESULTS, file)
        if os.path.exists(ref_path):
            my_path = os.path.join(MY_INPAINTED_RESULTS, file)
            stats = evaluate_3d_accuracy(my_path, ref_path,
                                         my_path.replace('.obj', '.mat'),
                                         ref_path.replace('.obj', '.mat'))
            stats['Filename'] = file
            comparison_data.append(stats)

# 3. Display Results
df_3d = pd.DataFrame(comparison_data)
print("=== 3D Mesh & Coefficient Comparison vs. Dataset Reference ===")
print(df_3d.mean(numeric_only=True))
display(df_3d.head())

import trimesh
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import os
import pandas as pd

# 1. Define Paths
# Based on your ls output and file structure
MY_INPAINTED_RESULTS = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000'
DATASET_REF_RESULTS = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000'

def evaluate_and_visualize_3d(my_obj, ref_obj, my_mat, ref_mat, show_mesh=False):
    results = {}

    # --- A. 3D Mesh Vertex Comparison ---
    if os.path.exists(my_obj) and os.path.exists(ref_obj):
        my_mesh = trimesh.load(my_obj)
        ref_mesh = trimesh.load(ref_obj)

        # Deep3D uses a fixed topology (same number of vertices)
        if len(my_mesh.vertices) == len(ref_mesh.vertices):
            # Calculate Euclidean distance for every vertex
            dist = np.linalg.norm(my_mesh.vertices - ref_mesh.vertices, axis=1)
            results['3D_Vertex_RMSE'] = np.sqrt(np.mean(dist**2))
            results['3D_Vertex_MAE'] = np.mean(dist)
            results['3D_Vertex_Max'] = np.max(dist)

            # Create Visualization Heatmap
            if show_mesh:
                # Normalize errors for coloring (0 to 1)
                norm_errors = (dist - dist.min()) / (dist.max() - dist.min() + 1e-8)
                colors = plt.cm.jet(norm_errors)[:, :3] # Use JET colormap

                error_mesh = trimesh.Trimesh(vertices=my_mesh.vertices,
                                             faces=my_mesh.faces,
                                             vertex_colors=colors)
                print(f"--- 3D Error Heatmap for {os.path.basename(my_obj)} ---")
                error_mesh.show()
        else:
            # Fallback for different topologies
            _, dists, _ = trimesh.proximity.closest_point(ref_mesh, my_mesh.vertices)
            results['Chamfer_Distance'] = np.mean(dists)

    # --- B. 3DMM Coefficient Comparison ---
    if os.path.exists(my_mat) and os.path.exists(ref_mat):
        my_data, ref_data = loadmat(my_mat), loadmat(ref_mat)
        for key in ['id', 'exp', 'tex']:
            if key in my_data and key in ref_data:
                v1, v2 = my_data[key].flatten(), ref_data[key].flatten()
                sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                results[f'{key}_Cosine_Sim'] = sim

    return results

# 2. Batch Comparison Logic
comparison_list = []
matched_files = [f for f in os.listdir(MY_INPAINTED_RESULTS) if f.endswith('.obj')]

print(f"Found {len(matched_files)} meshes to compare against reference dataset.\n")

for i, file in enumerate(matched_files):
    ref_path = os.path.join(DATASET_REF_RESULTS, file)

    if os.path.exists(ref_path):
        my_path = os.path.join(MY_INPAINTED_RESULTS, file)

        # Visualize the first 3 matches, then just collect data for the rest
        show_vis = True if i < 3 else False

        stats = evaluate_and_visualize_3d(
            my_path, ref_path,
            my_path.replace('.obj', '.mat'),
            ref_path.replace('.obj', '.mat'),
            show_mesh=show_vis
        )

        stats['Filename'] = file
        comparison_list.append(stats)

# 3. Final Summary Table
df_final = pd.DataFrame(comparison_list)
# Reorder columns to put Filename first
cols = ['Filename'] + [c for c in df_final.columns if c != 'Filename']
df_final = df_final[cols]

print("\n" + "="*50)
print("FINAL 3D ACCURACY SUMMARY (Inpainted vs. Reference)")
print("="*50)
print(df_final.mean(numeric_only=True))
display(df_final)

import trimesh
import numpy as np
import os
import pandas as pd
from scipy.io import loadmat

# --- CONFIGURATION: Match these to your actual folder names ---
# Your modified results (inpainted)
MY_RESULTS_DIR = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000'
# The GitHub repo's reference results
REF_RESULTS_DIR = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000'

def calculate_metrics(my_base_path, ref_base_path):
    metrics = {}

    # 1. 3D MESH COMPARISON (.obj)
    my_obj = my_base_path + '.obj'
    ref_obj = ref_base_path + '.obj'

    if os.path.exists(my_obj) and os.path.exists(ref_obj):
        m1 = trimesh.load(my_obj)
        m2 = trimesh.load(ref_obj)
        # Calculate Root Mean Square Error between vertices
        dist = np.linalg.norm(m1.vertices - m2.vertices, axis=1)
        metrics['Mesh_RMSE'] = np.sqrt(np.mean(dist**2))

    # 2. COEFFICIENT COMPARISON (.mat)
    my_mat = my_base_path + '.mat'
    ref_mat = ref_base_path + '.mat'

    if os.path.exists(my_mat) and os.path.exists(ref_mat):
        d1 = loadmat(my_mat)
        d2 = loadmat(ref_mat)
        # Compare 'id' (shape) and 'exp' (expression) coefficients
        for key in ['id', 'exp']:
            v1, v2 = d1[key].flatten(), d2[key].flatten()
            # Cosine Similarity: 1.0 is perfect, 0.0 is completely different
            cos_sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            metrics[f'{key}_Similarity'] = cos_sim

    return metrics

# --- RUN THE BATCH COMPARISON ---
all_stats = []
# Get list of filenames from your results
files = [f.replace('.obj', '') for f in os.listdir(MY_RESULTS_DIR) if f.endswith('.obj')]

for f in files:
    my_path = os.path.join(MY_RESULTS_DIR, f)
    ref_path = os.path.join(REF_RESULTS_DIR, f)

    if os.path.exists(ref_path + '.obj'):
        res = calculate_metrics(my_path, ref_path)
        res['Filename'] = f
        all_stats.append(res)

# Display as a Table
df = pd.DataFrame(all_stats)
print("\n=== FINAL COMPARISON REPORT ===")
if not df.empty:
    # Set 'Filename' as the first column
    df = df[['Filename', 'Mesh_RMSE', 'id_Similarity', 'exp_Similarity']]
    display(df)
    print("\nAverage Mesh Error (lower is better):", df['Mesh_RMSE'].mean())
    print("Average Identity Similarity (closer to 1.0 is better):", df['id_Similarity'].mean())
else:
    print("No matching files found between your results and the reference folder.")

import trimesh
import numpy as np
import os
import pandas as pd
from scipy.io import loadmat
import matplotlib.pyplot as plt

# --- PATH SETUP ---
# 1. Your custom pipeline results (with BiSeNet/Inpainting)
MY_DIR = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000'

# 2. The Reference results (The ones you found with !ls)
REF_DIR = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000'

def get_comparison_metrics(name):
    res = {'Filename': name}

    my_obj = os.path.join(MY_DIR, f"{name}.obj")
    ref_obj = os.path.join(REF_DIR, f"{name}.obj")
    my_mat = os.path.join(MY_DIR, f"{name}.mat")
    ref_mat = os.path.join(REF_DIR, f"{name}.mat")

    # A. 3D Mesh Metric (Root Mean Square Error)
    if os.path.exists(my_obj) and os.path.exists(ref_obj):
        m1 = trimesh.load(my_obj)
        m2 = trimesh.load(ref_obj)
        # Calculate distance between corresponding vertices
        diffs = np.linalg.norm(m1.vertices - m2.vertices, axis=1)
        res['3D_Vertex_RMSE'] = np.sqrt(np.mean(diffs**2))
        res['3D_Vertex_MAE'] = np.mean(diffs)

    # B. Coefficient Metric (Cosine Similarity)
    if os.path.exists(my_mat) and os.path.exists(ref_mat):
        d1, d2 = loadmat(my_mat), loadmat(ref_mat)
        for key in ['id', 'exp']: # Identity and Expression
            v1, v2 = d1[key].flatten(), d2[key].flatten()
            sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            res[f'{key}_Similarity'] = sim

    return res

# --- EXECUTION ---
# Get list of files that exist in both directories
my_files = {f.split('.')[0] for f in os.listdir(MY_DIR) if f.endswith('.obj')}
ref_files = {f.split('.')[0] for f in os.listdir(REF_DIR) if f.endswith('.obj')}
common_files = sorted(list(my_files.intersection(ref_files)))

print(f"Comparing {len(common_files)} matching files...")

final_results = []
for name in common_files:
    stats = get_comparison_metrics(name)
    final_results.append(stats)

# --- DISPLAY ---
df_metrics = pd.DataFrame(final_results)
if not df_metrics.empty:
    print("\n" + "="*40)
    print("3D RECONSTRUCTION ACCURACY REPORT")
    print("="*40)
    # Reorder columns for readability
    cols = ['Filename', '3D_Vertex_RMSE', 'id_Similarity', 'exp_Similarity']
    display(df_metrics[cols].head(20))

    print("\n--- Summary Averages ---")
    print(df_metrics.mean(numeric_only=True))
else:
    print("No matching files found! Check if filenames match in both folders.")

import trimesh
import numpy as np
import os
import pandas as pd
from scipy.io import loadmat
from scipy.spatial import cKDTree

# -------------------------------
# PATHS
# -------------------------------
MY_DIR = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/input_images_inpainted/epoch_20_000000'
REF_DIR = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples/epoch_20_000000'


# -------------------------------
# HELPER FUNCTIONS
# -------------------------------

def safe_cosine_similarity(v1, v2):
    """Numerically stable cosine similarity"""
    v1 = v1.flatten()
    v2 = v2.flatten()

    denom = (np.linalg.norm(v1) * np.linalg.norm(v2))
    if denom < 1e-8:
        return 0.0

    return float(np.dot(v1, v2) / denom)


def compute_chamfer_distance(points1, points2):
    """Symmetric Chamfer Distance"""
    tree1 = cKDTree(points1)
    tree2 = cKDTree(points2)

    dist1, _ = tree1.query(points2)
    dist2, _ = tree2.query(points1)

    chamfer = np.mean(dist1**2) + np.mean(dist2**2)
    return chamfer


def align_mesh_icp(source_mesh, target_mesh):
    """Align source → target using ICP"""
    matrix, aligned, cost = trimesh.registration.icp(
        source_mesh.vertices,
        target_mesh.vertices,
        scale=False
    )
    return aligned


# -------------------------------
# MAIN EVALUATION
# -------------------------------

def evaluate_3d_fidelity(name):
    my_obj = os.path.join(MY_DIR, f"{name}.obj")
    ref_obj = os.path.join(REF_DIR, f"{name}.obj")

    my_mat = os.path.join(MY_DIR, f"{name}.mat")
    ref_mat = os.path.join(REF_DIR, f"{name}.mat")

    metrics = {'Filename': name}

    # -------------------------------
    # A. GEOMETRY
    # -------------------------------
    if os.path.exists(my_obj) and os.path.exists(ref_obj):

        try:
            mesh_my = trimesh.load(my_obj, process=False)
            mesh_ref = trimesh.load(ref_obj, process=False)

            # Debug shapes (FIXED)
            print(f"{name} -> MY: {mesh_my.vertices.shape}, REF: {mesh_ref.vertices.shape}")

            # ICP alignment
            aligned_vertices = align_mesh_icp(mesh_my, mesh_ref)

            # Nearest neighbor distances
            tree = cKDTree(mesh_ref.vertices)
            dists, _ = tree.query(aligned_vertices)

            # RMSE
            rmse = np.sqrt(np.mean(dists**2))
            metrics['Mesh_RMSE'] = rmse

            # Chamfer
            chamfer = compute_chamfer_distance(aligned_vertices, mesh_ref.vertices)
            metrics['Chamfer_Distance'] = chamfer

            # -------------------------------
            # 🔥 NME (Normalized Mean Error)
            # -------------------------------
            mean_error = np.mean(dists)

            # Approximate normalization factor
            bbox_width = (
                np.max(mesh_ref.vertices[:, 0]) -
                np.min(mesh_ref.vertices[:, 0])
            )

            if bbox_width > 1e-8:
                nme = mean_error / bbox_width
            else:
                nme = 0.0

            metrics['NME'] = nme

        except Exception as e:
            print(f"⚠️ Mesh error for {name}: {e}")

    # -------------------------------
    # B. ID & EXP
    # -------------------------------
    if os.path.exists(my_mat) and os.path.exists(ref_mat):

        try:
            d1 = loadmat(my_mat)
            d2 = loadmat(ref_mat)

            for key in ['id', 'exp']:
                if key in d1 and key in d2:
                    metrics[f'{key}_Similarity'] = safe_cosine_similarity(d1[key], d2[key])

        except Exception as e:
            print(f"⚠️ MAT error for {name}: {e}")

    return metrics

# -------------------------------
# EXECUTION
# -------------------------------

common_files = [
    f.replace('.obj', '')
    for f in os.listdir(MY_DIR)
    if f.endswith('.obj') and os.path.exists(os.path.join(REF_DIR, f))
]

results = []

for name in common_files:
    stats = evaluate_3d_fidelity(name)
    results.append(stats)

df = pd.DataFrame(results)


# -------------------------------
# FINAL REPORT
# -------------------------------

if not df.empty:

    print("\n" + "="*60)
    print("🔥 CORRECTED 3D RECONSTRUCTION REPORT (ICP + CHAMFER)")
    print("="*60)

    display_cols = ['Filename', 'Chamfer_Distance', 'Mesh_RMSE', 'id_Similarity', 'exp_Similarity']
    display(df[display_cols])

    print("\n📊 OVERALL AVERAGES:")
    print(f"Chamfer Distance: {df['Chamfer_Distance'].mean():.6f} (Lower is better)")
    print(f"Mesh RMSE:        {df['Mesh_RMSE'].mean():.6f} (Lower is better)")
    print(f"Identity Sim:     {df['id_Similarity'].mean():.6f} (Higher is better)")
    print(f"Expression Sim:   {df['exp_Similarity'].mean():.6f} (Higher is better)")

else:
    print("❌ No matching files found.")

import os
import cv2
import numpy as np
from tqdm import tqdm

INPUT_DIR = './datasets/examples'
OUTPUT_DIR = './datasets/examples_inpainted'

os.makedirs(OUTPUT_DIR, exist_ok=True)

def simple_inpaint(image):
    # Dummy mask (replace with BiSeNet later)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY_INV)

    inpainted = cv2.inpaint(image, mask, 3, cv2.INPAINT_TELEA)
    return inpainted

for img_name in tqdm(os.listdir(INPUT_DIR)):
    if not img_name.endswith(('.jpg', '.png')):
        continue

    img_path = os.path.join(INPUT_DIR, img_name)
    img = cv2.imread(img_path)

    output = simple_inpaint(img)

    cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), output)

!rm -rf ./datasets/examples_backup
!mv ./datasets/examples ./datasets/examples_backup
!mv ./datasets/examples_inpainted ./datasets/examples

!ls ./datasets/examples

!cp -r ./datasets/examples ./datasets/examples_original

import matplotlib.pyplot as plt

img = cv2.imread('./datasets/examples_inpainted/000002.jpg')
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Inpainted result")
plt.axis('off')

import os
import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import sys

# --- 1. SETUP PATHS & DEVICE ---
# Ensure face-parsing.PyTorch is in your path (adjust if different)
sys.path.insert(0, '/content/Deep3DFaceRecon_pytorch/face-parsing.PyTorch')
from model import BiSeNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_DIR = './datasets/examples_original'
OUTPUT_DIR = './datasets/examples_inpainted'
WEIGHTS_PATH = '/content/Deep3DFaceRecon_pytorch/face-parsing.PyTorch/res/cp/79999_iter.pth'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. LOAD BISENET MODEL ---
net = BiSeNet(n_classes=19)
state = torch.load(WEIGHTS_PATH, map_location=device)
if all(k.startswith('module.') for k in state.keys()):
    state = {k[7:]: v for k, v in state.items()}
net.load_state_dict(state)
net.to(device)
net.eval()

# --- 3. HELPER FUNCTIONS (From your file) ---
bisenet_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Labels to keep as "Face" (everything else is occlusion)
FACE_LABELS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

@torch.no_grad()
def get_segmentation(net, img_bgr, device):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    inp = bisenet_transform(pil).unsqueeze(0).to(device)
    out = net(inp)[0]
    seg = out.squeeze(0).argmax(0).cpu().numpy().astype(np.uint8)
    seg = cv2.resize(seg, (img_bgr.shape[1], img_bgr.shape[0]), interpolation=cv2.INTER_NEAREST)
    return seg

def make_face_mask(seg_map):
    mask = np.zeros_like(seg_map, dtype=np.uint8)
    for lbl in FACE_LABELS:
        mask[seg_map == lbl] = 1
    # Smooth the mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

# --- 4. MAIN PROCESSING LOOP ---
print("🚀 Starting BiSeNet + Inpainting process...")
for img_name in tqdm(os.listdir(INPUT_DIR)):
    if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path = os.path.join(INPUT_DIR, img_name)
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: continue

    # Step A: Get Face Segmentation
    seg_map = get_segmentation(net, img_bgr, device)
    face_mask = make_face_mask(seg_map).astype(bool)

    # Step B: Create Occlusion Mask (Inverse of face mask)
    # Regions NOT identified as face parts are treated as occlusions to be inpainted
    occlusion_mask = (~face_mask).astype(np.uint8) * 255

    # Step C: Inpainting
    # This fills in the "non-face" areas using surrounding face texture
    inpainted = cv2.inpaint(img_bgr, occlusion_mask, 3, cv2.INPAINT_TELEA)

    # Save Result
    cv2.imwrite(os.path.join(OUTPUT_DIR, img_name), inpainted)

print(f"✅ Finished! Inpainted images saved to: {OUTPUT_DIR}")

!mv ./datasets/examples ./datasets/examples_original
!mv ./datasets/examples_inpainted ./datasets/examples

import os
import cv2
import torch
import numpy as np
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import sys

# --- 1. SETUP & DIRECTORIES ---
# Make sure we have a clean start for folders
!mkdir -p ./datasets/examples_original
!mkdir -p ./datasets/examples_inpainted

# Assuming your original images are currently in ./datasets/examples/
INPUT_DIR = './datasets/examples'
ORIG_SAVE_DIR = './datasets/examples_original'
INPAINT_SAVE_DIR = './datasets/examples_inpainted'

# Copy originals to the 'original' folder for safe keeping/comparison
!cp ./datasets/examples/*.jpg ./datasets/examples_original/ 2>/dev/null || :
!cp ./datasets/examples/*.png ./datasets/examples_original/ 2>/dev/null || :

# --- 2. LOAD BISENET (Keep your existing loading code here) ---
# [Insert your BiSeNet loading code here...]

# --- 3. RUN THE INPAINTING LOOP ---
print("🚀 Running BiSeNet + Inpainting...")
for img_name in tqdm(os.listdir(ORIG_SAVE_DIR)):
    if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img_path = os.path.join(ORIG_SAVE_DIR, img_name)
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: continue

    # BiSeNet Logic
    seg_map = get_segmentation(net, img_bgr, device)
    face_mask = make_face_mask(seg_map).astype(bool)
    occlusion_mask = (~face_mask).astype(np.uint8) * 255

    # Inpainting
    inpainted = cv2.inpaint(img_bgr, occlusion_mask, 3, cv2.INPAINT_TELEA)

    # SAVE TO INPAINTED FOLDER
    cv2.imwrite(os.path.join(INPAINT_SAVE_DIR, img_name), inpainted)

print("✅ Inpainting Complete.")

import os
import cv2
import face_alignment
import numpy as np
from tqdm import tqdm

# Initialize the landmark detector
fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, flip_input=False, device='cuda')

def generate_landmarks(img_folder):
    detect_dir = os.path.join(img_folder, 'detections')
    os.makedirs(detect_dir, exist_ok=True)

    image_files = [f for f in os.listdir(img_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print(f"📍 Generating landmarks for {img_folder}...")
    for img_name in tqdm(image_files):
        img_path = os.path.join(img_folder, img_name)

        # FIX: Use OpenCV instead of face_alignment.io
        img = cv2.imread(img_path)
        if img is None:
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # FA requires RGB

        preds = fa.get_landmarks(img_rgb)

        if preds is not None:
            lm68 = preds[0]
            # Deep3D expects these 5 specific points for alignment:
            lm5 = np.zeros((5, 2))
            lm5[0] = np.mean(lm68[36:42], axis=0) # Left Eye
            lm5[1] = np.mean(lm68[42:48], axis=0) # Right Eye
            lm5[2] = lm68[30]                      # Nose Tip
            lm5[3] = lm68[48]                      # Left Mouth
            lm5[4] = lm68[54]                      # Right Mouth

            txt_name = os.path.splitext(img_name)[0] + '.txt'
            np.savetxt(os.path.join(detect_dir, txt_name), lm5, fmt='%.3f')

# Run for both folders
generate_landmarks('./datasets/examples_original')
generate_landmarks('./datasets/examples_inpainted')

!ls

!ls datasets

!ls
!ls datasets

!pwd
!ls

!find /content -type d -name "examples_inpainted"
!find /content -type d -name "examples_original"

# Commented out IPython magic to ensure Python compatibility.
# %cd /content/Deep3DFaceRecon_pytorch

# Inpainted
!python test.py \
  --name face_recon_feat0.2_augment \
  --epoch 20 \
  --img_folder datasets/examples_inpainted

# Original
!python test.py \
  --name face_recon_feat0.2_augment \
  --epoch 20 \
  --img_folder datasets/examples_original

import trimesh
import numpy as np
import os
import pandas as pd
from scipy.spatial import cKDTree

ORIG_DIR = 'checkpoints/face_recon_feat0.2_augment/results/examples_original/epoch_20_000000'
INP_DIR  = 'checkpoints/face_recon_feat0.2_augment/results/examples_inpainted/epoch_20_000000'

def align_icp(src, tgt):
    matrix, aligned, cost = trimesh.registration.icp(
        src.vertices, tgt.vertices, scale=False
    )
    return aligned

def compute_rmse(a, b):
    tree = cKDTree(b)
    dists, _ = tree.query(a)
    return np.sqrt(np.mean(dists**2))

results = []

files = [f.replace('.obj','') for f in os.listdir(ORIG_DIR) if f.endswith('.obj')]

for name in files:
    orig_path = os.path.join(ORIG_DIR, name + '.obj')
    inp_path  = os.path.join(INP_DIR, name + '.obj')

    if not os.path.exists(inp_path):
        continue

    mesh_orig = trimesh.load(orig_path, process=False)
    mesh_inp  = trimesh.load(inp_path, process=False)

    aligned = align_icp(mesh_inp, mesh_orig)

    rmse = compute_rmse(aligned, mesh_orig.vertices)

    results.append({
        'Filename': name,
        'RMSE (Inpainted vs Original)': rmse
    })

df = pd.DataFrame(results)

print(df)
print("\nAverage RMSE:", df.iloc[:,1].mean())

import trimesh
import numpy as np
import os
import pandas as pd
from scipy.io import loadmat

# Paths
PATH_ORIG = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples_original/epoch_20_000000'
PATH_INPAINT = '/content/Deep3DFaceRecon_pytorch/checkpoints/face_recon_feat0.2_augment/results/examples_inpainted/epoch_20_000000'

# -----------------------------
# NME Calculation
# -----------------------------
def compute_nme(v1, v2):
    errors = np.linalg.norm(v1 - v2, axis=1)

    bbox_min = np.min(v1, axis=0)
    bbox_max = np.max(v1, axis=0)
    diag = np.linalg.norm(bbox_max - bbox_min)

    if diag == 0:
        return np.nan

    return np.mean(errors) / diag


# -----------------------------
# Main Comparison Function
# -----------------------------
def compare_results(name):
    res = {'Filename': name}

    orig_obj = os.path.join(PATH_ORIG, f"{name}.obj")
    inpaint_obj = os.path.join(PATH_INPAINT, f"{name}.obj")
    orig_mat = os.path.join(PATH_ORIG, f"{name}.mat")
    inpaint_mat = os.path.join(PATH_INPAINT, f"{name}.mat")

    # --- Mesh Metrics ---
    if os.path.exists(orig_obj) and os.path.exists(inpaint_obj):
        m1 = trimesh.load(orig_obj)
        m2 = trimesh.load(inpaint_obj)

        v1, v2 = m1.vertices, m2.vertices

        # Sanity check
        if v1.shape != v2.shape:
            print(f"[WARNING] Shape mismatch in {name}")
            res['Mesh_RMSE'] = np.nan
            res['NME'] = np.nan
        else:
            dist = np.linalg.norm(v1 - v2, axis=1)

            # RMSE
            res['Mesh_RMSE'] = np.sqrt(np.mean(dist**2))

            # NME
            res['NME'] = compute_nme(v1, v2)

    # --- Parameter Similarity ---
    if os.path.exists(orig_mat) and os.path.exists(inpaint_mat):
        mat1 = loadmat(orig_mat)
        mat2 = loadmat(inpaint_mat)

        for key in ['id', 'exp']:
            if key in mat1 and key in mat2:
                v1 = mat1[key].flatten()
                v2 = mat2[key].flatten()

                denom = (np.linalg.norm(v1) * np.linalg.norm(v2))
                if denom == 0:
                    sim = np.nan
                else:
                    sim = np.dot(v1, v2) / denom

                res[f'{key}_Similarity'] = sim

    return res


# -----------------------------
# Run Evaluation
# -----------------------------
filenames = [f.replace('.obj', '') for f in os.listdir(PATH_INPAINT) if f.endswith('.obj')]
all_stats = [compare_results(f) for f in filenames]

df = pd.DataFrame(all_stats)

print("\n" + "="*60)
print("FULL 3D RECONSTRUCTION REPORT")
print("="*60)
display(df)

print("\nOVERALL MEAN:")
print(df.mean(numeric_only=True))

print("\nOVERALL MEDIAN:")
print(df.median(numeric_only=True))


# -----------------------------
# Failure Analysis
# -----------------------------
FAIL_THRESHOLD = 0.1

df_failures = df[df['NME'] >= FAIL_THRESHOLD]
df_filtered = df[df['NME'] < FAIL_THRESHOLD]

print("\n" + "="*60)
print("FAILURE CASES (NME >= 0.1)")
print("="*60)
display(df_failures)

print("\n" + "="*60)
print("FILTERED RESULTS (GOOD CASES ONLY)")
print("="*60)
display(df_filtered)

# Stats
num_total = len(df)
num_filtered = len(df_filtered)

print(f"\nRemoved {num_total - num_filtered} failure cases out of {num_total}")
print(f"Retained {num_filtered} valid samples")

print("\nFILTERED MEAN:")
print(df_filtered.mean(numeric_only=True))

print("\nFILTERED MEDIAN:")
print(df_filtered.median(numeric_only=True))

